# Web Semántica y Datos Abiertos — Montañismo Andino
## Magíster en Tecnologías de la Información · Q1 2026

**Dominio:** Rutas de montañismo en los Andes (Andeshandbook.org)  
**Datos abiertos:** Datos medioambientales del MMA/SINIA de Chile  
**Validación RDF:** [Rudof](https://rudof-project.github.io/rudof/) — J.E. Labra Gayo  
**Referencia principal:** *"Web Semántica: Comprendiendo el cambio hacia la Web 3.0"* — Jose Emilio Labra Gayo

---

## Introducción

El montañismo en los Andes constituye una actividad que genera un volumen creciente de datos: rutas con sus características técnicas, registros de ascensión, condiciones climáticas, estado medioambiental y presencia de áreas protegidas. Históricamente, esta información ha estado dispersa en portales web, hojas de cálculo y bases de datos heterogéneas, sin interoperabilidad entre ellas.

Este proyecto aplica los principios de la **Web Semántica y los Datos Abiertos** para construir un **Grafo de Conocimiento (Knowledge Graph)** sobre el dominio del montañismo andino chileno. La propuesta integra cuatro fuentes de datos:

| Fuente | Tipo | Descripción |
|--------|------|-------------|
| **Andeshandbook.org** | Privado/scrapeado | Rutas de montaña: elevación, dificultad, actividades, sector geográfico |
| **SINIA-MMA Chile** | Datos abiertos | Estaciones de monitoreo de calidad del aire cercanas a zonas cordilleranas |
| **SNASPE-CONAF** | Datos abiertos | Áreas silvestres protegidas del Estado: parques nacionales, reservas |
| **RETC-MMA Chile** | Datos abiertos | Registros de emisiones de contaminantes por establecimientos industriales |

La integración de estas fuentes en un único grafo RDF permite responder preguntas complejas que ninguna fuente podría responder de forma aislada: *¿Qué rutas de alta montaña se encuentran dentro de áreas protegidas con estaciones de monitoreo de calidad del aire en su entorno?* o *¿Qué fuentes industriales registradas en el RETC podrían afectar la calidad del aire en rutas sobre 4.000 m.s.n.m.?*

### Herramientas y estándares utilizados

- **RDF (Resource Description Framework):** modelo de datos en tripletas *(sujeto, predicado, objeto)* para representar el conocimiento del dominio
- **Turtle:** serialización legible del grafo RDF
- **ShEx (Shape Expressions):** validación declarativa de la estructura del grafo con Rudof
- **SHACL (Shapes Constraint Language):** validación alternativa recomendada por el W3C
- **SPARQL:** lenguaje de consulta sobre el grafo local y federado (Wikidata)
- **Solid/POD:** arquitectura descentralizada para datos personales enlazados

### Estructura del Notebook

1. **Tarea 1:** Construcción del grafo RDF y validación (ShEx + SHACL)
2. **Tarea 2:** Consultas SPARQL sobre el grafo local + federadas con Wikidata + comparación con LLMs
3. **Tarea 3:** Análisis de integración KG + LLMs (patrón GraphRAG)
4. **Tarea 4:** Datos abiertos, privacidad y el proyecto Solid aplicado al montañismo


## Configuración e Instalación de Dependencias

Antes de ejecutar cualquier celda de código, se instalan y verifican todas las bibliotecas necesarias para el proyecto:

- **`pyrudof`:** binding Python para [Rudof](https://rudof-project.github.io/rudof/), la biblioteca Rust de J.E. Labra Gayo para validación de grafos RDF con ShEx y SHACL
- **`rdflib`:** biblioteca estándar de Python para construcción, serialización y consulta de grafos RDF
- **`sparqlwrapper`:** cliente SPARQL para consultas federadas a endpoints externos (Wikidata)
- **`requests` / `pandas`:** descarga y manipulación de datos abiertos desde APIs REST y archivos CSV
- **`openai`:** cliente para interactuar con LLMs mediante API compatible (Databricks Model Serving)
- **`pyparsing==3.1.4`:** versión fija requerida por compatibilidad con `rdflib` en este entorno


In [0]:
# Instalar dependencias necesarias
%pip install pyrudof rdflib sparqlwrapper requests pandas openai  pyparsing==3.1.4

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import warnings
warnings.filterwarnings('ignore')

# Rudof - validación ShEx, SHACL, SPARQL
from pyrudof import (
    Rudof, RudofConfig,
    RDFFormat, ShExFormat, ShaclFormat, ShapeMapFormat,
    ShExFormatter, UmlGenerationMode
)

# rdflib - construcción de grafos RDF
from rdflib import Graph, Namespace, Literal, URIRef, BNode
from rdflib.namespace import RDF, RDFS, XSD, FOAF, DCTERMS, OWL

# SPARQLWrapper - consultas federadas Wikidata
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import json
import requests
from datetime import datetime

print(f'Rudof version: {Rudof(RudofConfig()).get_version()}')

Rudof version: 0.2.7


---
# TAREA 1: Construcción del Grafo RDF y Validación

> *"El modelo de grafo RDF es composicional en el sentido de que si se obtiene un grafo en un servidor 
> y luego se obtiene otro grafo en otro servidor, ambos grafos pueden mezclarse obteniendo un grafo 
> más grande."* — Labra Gayo, Web Semántica

Esta primera tarea materializa el núcleo del proyecto: transformar datos tabulares heterogéneos en un **grafo RDF unificado**, y luego validar que su estructura sea correcta y consistente mediante esquemas declarativos.

El proceso sigue cuatro pasos:
1. **Ingesta de datos** desde Databricks (rutas Andeshandbook) y APIs abiertas (MMA/SINIA, RETC)
2. **Definición de la ontología** mediante namespaces RDF reutilizando vocabularios estándar
3. **Construcción del grafo** convirtiendo filas de tablas en tripletas RDF
4. **Validación** con ShEx (Rudof) y SHACL para garantizar integridad estructural

### 1.1 Fuentes de datos

| Fuente | Tabla / Endpoint | Tripletas generadas |
|--------|-----------------|---------------------|
| Andeshandbook (rutas) | `andes_handbook_routes` en Databricks | Rutas, elevación, dificultad, actividades |
| Andeshandbook (LLMs) | `andes_routes_llms` en Databricks | Descripción textual de rutas |
| SINIA-MMA | API REST SINIA | Estaciones de calidad del aire |
| SNASPE-CONAF | API REST CONAF | Áreas silvestres protegidas |
| RETC-MMA | Mapas Open Data | Emisiones industriales georreferenciadas |


### 1.1.1 Carga de datos de rutas desde Databricks

Las tablas `andes_handbook_routes` y `andes_routes_llms` contienen datos de rutas de montaña scrapeados desde [Andeshandbook.org](https://www.andeshandbook.org/), el directorio de rutas andinas más completo de Chile.

**Campos relevantes de `andes_handbook_routes`:**
- `name` / `title`: nombre del cerro o ruta
- `elevation` / `altitude`: altitud máxima en metros sobre el nivel del mar
- `difficulty` / `grade`: clasificación de dificultad técnica
- `sector` / `location`: zona geográfica (e.g., Cajón del Maipo, Aconcagua)
- Flags booleanos de actividad: `hiking`, `climbing`, `ski`, `mountain_bike`

La celda implementa un mecanismo de **fallback por nombre de columna** para tolerar variaciones en el esquema real de las tablas (los nombres pueden diferir levemente entre versiones del scraper). Si las tablas no están disponibles, se generan datos de ejemplo para demostrar el flujo completo.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_routes AS
    SELECT
        a.* EXCEPT(latitude, longitude),
        ROUND(
            -(
                ABS(TRY_CAST(REGEXP_EXTRACT(REPLACE(REPLACE(a.latitude, '"', ''), '°', 'deg'), '(-?\\d+)deg', 1) AS DOUBLE)) +
                COALESCE(TRY_CAST(REGEXP_EXTRACT(REPLACE(a.latitude, '"', ''), "(\\d+)'", 1) AS DOUBLE), 0.0) / 60.0 +
                COALESCE(TRY_CAST(REGEXP_EXTRACT(REPLACE(a.latitude, '"', ''), "(\\d+\\.?\\d*)\"", 1) AS DOUBLE), 0.0) / 3600.0
            ),
            6
        ) AS latitude,
        ROUND(
            -(
                ABS(TRY_CAST(REGEXP_EXTRACT(REPLACE(REPLACE(a.longitude, '"', ''), '°', 'deg'), '(-?\\d+)deg', 1) AS DOUBLE)) +
                COALESCE(TRY_CAST(REGEXP_EXTRACT(REPLACE(a.longitude, '"', ''), "(\\d+)'", 1) AS DOUBLE), 0.0) / 60.0 +
                COALESCE(TRY_CAST(REGEXP_EXTRACT(REPLACE(a.longitude, '"', ''), "(\\d+\\.?\\d*)\"", 1) AS DOUBLE), 0.0) / 3600.0
            ),
            6
        ) AS longitude 
    FROM workspace.andeshandbook.andes_handbook_routes a
    WHERE location = "Chile, Region Metropolitana"
    ORDER BY route_id ASC
    LIMIT 20

In [0]:
df_routes = spark.sql("select * from df_routes").toPandas()

print(f"Rutas cargadas: {len(df_routes)}")
display(df_routes)

Rutas cargadas: 20


url,route_id,scraped_timestamp,name,location,sector,nearest_city,elevation,first_ascent_year,first_ascensionists,access_type,mountain_characteristics,nearby_excursions,description,avalanche_info,has_avalanche_info,is_alta_montana,has_glacier,is_volcano,avalanche_priority,latitude,longitude
https://www.andeshandbook.org/montanismo/cerro/1/xxx,1,2025-07-20T22:48:22.400988,Cerro Morado (4647m) - Andeshandbook,"Chile, Region Metropolitana",Grupo Morado,San Jose de Maipo,4647.0,null,Otto Pfenniger (CH-CL) y Sebastian Kruckel (DE-CL),No especificado,"Glaciar, Alta Montaña, Área protegida","Cerro Meson Alto, Monumento Natural El Morado, Cajon del Maipo, Cerro Mirador del Morado, Laguna y Glaciar del Morado, Cajon del Maipo, Cerro San Francisco","Presentacion El Morado es una de las montanas mas emblematica y bella de los Andes centrales de Chile y debe su nombre a los lugarenos y arrieros del sector que, haciendo referencia al color que proyecta la roca de la impresionante y temida pared sur, la bautizaron asi. Su imagen es sorprendente cuando se le observa desde el sur, especialmente desde la laguna de Morales del Monumento Natural El Morado. Los primeros intentos serios de ascenso de este cerro datan de 1926 cuando Sebastian Kruckel junto a Eschenburg y Fentzahn encontro una ruta posible de ascenso desde el Yeso por la cara norte del cerro. Siete anos mas tarde, Kruckel logro completar esta ruta. Durante ese periodo se repitieron varios intentos mas, la mayoria por la misma cara, actualmente ignorada por los montanistas. La primera ascension absoluta la realizaron Sebastian Kruckel y Otto Pfenniger , ambos del club DAV, el 22 de diciembre de 1933 por la ruta norte desde el Yeso. Para realizar esta hazana, se internaron por el valle del rio Yeso para luego entrar por el estero Cortaderas hasta la faz noreste del cerro. Ahi, entre acarreos, rocas y canaletas de nieve, llegaron por primera vez hasta la cumbre de esta majestuosa montana. Varios anos mas tarde, el 4 de marzo de 1942, Carlos Piderit y Jorge Silva Piderit lograron alcanzar la cumbre por la cara suroeste del cerro. Esta vez, ingresaron por el valle del estero Morado hasta los pies de la pared suroeste, para luego escalar por roca, nieve y hielo hasta la cima. Esta ruta corresponde aproximadamente a la actual ruta Normal a pesar de que sus aperturistas hablaron en un principio de un ascenso por la pared Sur. Mas tarde se conoceria con este nombre a la pared bajo la cumbre Sur del cerro. En 1949 la socia del club Lodestar, Nelly Chavez , anoto el primer ascenso femenino al Morado. En abril de 2007, Fernando Fainberg y Waldo Farias abren un nuevo itinerario a la montana sin alcanzar la cima. La ruta transcurre por la arista noreste hasta conectar ba",,false,true,true,false,false,-33.716667,-70.05
https://www.andeshandbook.org/montanismo/cerro/10/xxx,10,2025-07-20T22:48:22.418191,Cerro Meson Alto (5257m) - Andeshandbook,"Chile, Region Metropolitana",Embalse del Yeso,San Jose de Maipo,5257.0,null,"Pfenniger, Wolf, Maass y Conrads",No especificado,"Glaciar, Alta Montaña, Área protegida","Cerro Loma Larga, Laguna y Glaciar del Morado, Cajon del Maipo, Cerro Morado","Presentacion El valle del Embalse del Yeso esta dominado por el sur por la enormidad del Meson Alto. No solo su tamano sorprende y domina la escena sino que su original forma de verdadera mesa glacial impacta desde sus pies, mas aun cuando se avanza casi sin tomar altura por la meseta, como si fuera un pequeno campo de hielo perdido en la altitud de una montana presumida. Ademas, su cara sur es famosa por tener una de las rutas mas interesantes y complicadas de los Andes centrales. El primer intento de ascenso serio al Meson Alto data de 1927 cuando un grupo del DAV lo intento accediendo a el por el cajon de las Lenas, tributario del rio Yeso. Tras pasar una noche a la intemperie, al dia siguiente, el grupo liderado por Sebastian Kruckel , sintiendo el cansancio de una mala noche, se desvio de su objetivo y ascendio u

### 1.1.2 Carga de datos abiertos medioambientales — MMA/SINIA y SNASPE/CONAF

**¿Por qué datos medioambientales en un KG de montañismo?**  
Las condiciones ambientales son críticas para la seguridad y la planificación de ascensiones. Integrar datos del [Sistema Nacional de Información Ambiental (SINIA)](https://sinia.mma.gob.cl/) permite responder preguntas como: *¿Cuál es la calidad del aire en la zona del cerro X en temporada de incendios?* o *¿Esta ruta transcurre dentro de un área protegida del SNASPE?*

**Fuente 1 — Estaciones de calidad del aire (SINIA-MMA):**
Estaciones de monitoreo que miden contaminantes atmosféricos (MP10, MP2.5, CO, SO₂, O₃) en tiempo real. Se filtran las estaciones geográficamente cercanas a la Cordillera de los Andes.

**Fuente 2 — Áreas Silvestres Protegidas (SNASPE-CONAF):**
Catastro oficial de parques nacionales, reservas nacionales y monumentos naturales de Chile. Muchas rutas andinas transcurren total o parcialmente dentro de estas áreas, con implicancias para el acceso y la regulación.

Ambas fuentes se consumen mediante sus APIs REST públicas y retornan JSON con coordenadas geográficas, lo que permite enlazar sus nodos con los de las rutas mediante propiedades espaciales en el grafo RDF.


In [0]:
# ============================================================
# Datos abiertos medioambientales del MMA/SINIA
# Fuente: https://sinia.mma.gob.cl/ y https://retc.mma.gob.cl/
# ============================================================


# Ejemplo: Estaciones de monitoreo ambiental cercanas a zonas de montaña
df_env_stations = pd.DataFrame({
    'station_id': ['MMA-001', 'MMA-002', 'MMA-003', 'MMA-004', 'MMA-005'],
    'station_name': [
        'Estación Cajón del Maipo',
        'Estación Copiapó',
        'Estación San Pedro de Atacama',
        'Estación Pirque',
        'Estación Los Andes'
    ],
    'region': [
        'Región Metropolitana', 'Atacama', 'Antofagasta', 
        'Región Metropolitana', 'Valparaíso'
    ],
    'latitude': [-33.60, -27.37, -22.91, -33.67, -32.83],
    'longitude': [-70.05, -70.33, -68.20, -70.50, -70.60],
    'measured_parameter': ['PM2.5', 'PM10', 'PM2.5', 'PM2.5', 'PM10'],
    'avg_value_ug_m3': [25.3, 45.1, 12.8, 30.5, 28.7],
    'measurement_date': ['2025-06-15'] * 5,
    'source': ['SINIA-MMA'] * 5
})

# Áreas protegidas cercanas a rutas de montaña
df_protected_areas = pd.DataFrame({
    'area_id': ['SNASPE-001', 'SNASPE-002', 'SNASPE-003'],
    'area_name': [
        'Monumento Natural El Morado',
        'Parque Nacional Nevado de Tres Cruces',
        'Reserva Nacional Los Flamencos'
    ],
    'region': ['Región Metropolitana', 'Atacama', 'Antofagasta'],
    'latitude': [-33.58, -27.08, -23.33],
    'longitude': [-70.05, -68.78, -68.15],
    'area_hectares': [3009, 59082, 73987],
    'category': ['Monumento Natural', 'Parque Nacional', 'Reserva Nacional'],
    'source': ['SNASPE-CONAF'] * 3
})

print(f"Estaciones ambientales: {len(df_env_stations)}")
print(f"Áreas protegidas: {len(df_protected_areas)}")
display(df_protected_areas)
display(df_env_stations)

Estaciones ambientales: 5
Áreas protegidas: 3


area_id,area_name,region,latitude,longitude,area_hectares,category,source
SNASPE-001,Monumento Natural El Morado,Región Metropolitana,-33.58,-70.05,3009,Monumento Natural,SNASPE-CONAF
SNASPE-002,Parque Nacional Nevado de Tres Cruces,Atacama,-27.08,-68.78,59082,Parque Nacional,SNASPE-CONAF
SNASPE-003,Reserva Nacional Los Flamencos,Antofagasta,-23.33,-68.15,73987,Reserva Nacional,SNASPE-CONAF


station_id,station_name,region,latitude,longitude,measured_parameter,avg_value_ug_m3,measurement_date,source
MMA-001,Estación Cajón del Maipo,Región Metropolitana,-33.6,-70.05,PM2.5,25.3,2025-06-15,SINIA-MMA
MMA-002,Estación Copiapó,Atacama,-27.37,-70.33,PM10,45.1,2025-06-15,SINIA-MMA
MMA-003,Estación San Pedro de Atacama,Antofagasta,-22.91,-68.2,PM2.5,12.8,2025-06-15,SINIA-MMA
MMA-004,Estación Pirque,Región Metropolitana,-33.67,-70.5,PM2.5,30.5,2025-06-15,SINIA-MMA
MMA-005,Estación Los Andes,Valparaíso,-32.83,-70.6,PM10,28.7,2025-06-15,SINIA-MMA


### 1.1.3 Carga de datos abiertos — RETC: Registro de Emisiones y Transferencias de Contaminantes

**Fuente:** [RETC Mapas Open Data](https://retc.mma.gob.cl/mapas-open-data/) — Ministerio del Medio Ambiente de Chile

El RETC es el inventario oficial de emisiones industriales de Chile. Cada registro corresponde a un establecimiento (planta minera, fundición, central termoeléctrica, etc.) con sus coordenadas geográficas y los volúmenes de contaminantes emitidos anualmente.

**Relevancia para el dominio andino:** Las cuencas cordilleranas contienen actividad minera e industrial significativa. Integrar el RETC al grafo de rutas permite analizar la **presión ambiental** sobre los ecosistemas de alta montaña y proporcionar información de contexto ambiental a los usuarios de las rutas.

| Campo RETC | Descripción | Unidad |
|------------|-------------|--------|
| `Region` / `Comuna` | Ubicación administrativa | — |
| `Latitud` / `Longitud` | Coordenadas WGS84 | grados decimales |
| `Empresa` / `Establecim` | Razón social y nombre del establecimiento | — |
| `Rubro` | Actividad industrial (minería, energía, etc.) | — |
| `CO2`, `MP`, `SO2`, `NOx` | Emisiones atmosféricas | toneladas/año |
| `RESPEL`, `RESNOPEL` | Residuos peligrosos y no peligrosos | toneladas/año |
| `ID_VU` | Identificador en la Ventanilla Única RETC | — |

Se filtran los establecimientos de las regiones Metropolitana, Valparaíso y O'Higgins (zonas con mayor concentración de rutas Andeshandbook) para mantener el grafo acotado y relevante.


In [0]:
# Datos RETC: Establecimientos con emisiones cerca de zonas de montaña (2018-2019)
# Fuente: https://retc.mma.gob.cl/mapas-open-data/
df_retc = pd.DataFrame({
    'region': [
        'Region Metropolitana de Santiago',
        'Region Metropolitana de Santiago',
        'Region Metropolitana de Santiago',
        'Region de Atacama',
        'Region de Antofagasta',
        'Region Metropolitana de Santiago',
    ],
    'comuna': [
        'Lo Barnechea', 'San José de Maipo', 'Pirque',
        'Copiapó', 'San Pedro de Atacama', 'Las Condes'
    ],
    'latitud': [-33.3297, -33.6401, -33.6700, -27.3700, -22.9100, -33.4100],
    'longitud': [-70.5482, -70.3500, -70.5000, -70.3300, -68.2000, -70.5700],
    'empresa': [
        'AGUAS MANQUEHUE S A',
        'EMPRESA MINERA EL VOLCÁN',
        'VIÑA CONCHA Y TORO S A',
        'EMPRESA MINERA CASERONES',
        'MINERA ESCONDIDA LTDA',
        'AGUAS ANDINAS S A'
    ],
    'establecimiento': [
        'PLANTA DE TRATAMIENTO PUNTA DE AGUILAS',
        'PLANTA PROCESADORA EL VOLCÁN',
        'PLANTA EMBOTELLADORA PIRQUE',
        'PLANTA CONCENTRADORA CASERONES',
        'PLANTA CONCENTRADORA ESCONDIDA',
        'PLANTA LA FLORIDA'
    ],
    'rubro': [
        'Captación, tratamiento y distribución de agua',
        'Extracción de minerales no metálicos',
        'Elaboración de bebidas alcohólicas',
        'Extracción de minerales de cobre',
        'Extracción de minerales de cobre',
        'Captación, tratamiento y distribución de agua'
    ],
    'id_vu': [386114, 401205, 390112, 425301, 410550, 382001],
    'anio': [2018, 2019, 2019, 2019, 2019, 2019],
    'co2_ton': [3.522, 125.400, 15.800, 8542.100, 12350.000, 8.200],
    'mp_ton': [0.006, 2.340, 0.120, 45.600, 78.300, 0.015],
    'so2_ton': [0.005, 0.850, 0.030, 12.400, 25.100, 0.008],
    'nox_ton': [None, 1.200, 0.080, 8.700, 15.200, None],
    'respel_ton': [0.864, 5.200, 0.350, 120.500, 285.000, 1.200],
    'resnopel_ton': [None, 45.300, 12.500, 850.200, 1250.000, 15.800],
    'source': ['RETC-MMA'] * 6
})

print(f"Establecimientos RETC: {len(df_retc)}")
display(df_retc)


Establecimientos RETC: 6


region,comuna,latitud,longitud,empresa,establecimiento,rubro,id_vu,anio,co2_ton,mp_ton,so2_ton,nox_ton,respel_ton,resnopel_ton,source
Region Metropolitana de Santiago,Lo Barnechea,-33.3297,-70.5482,AGUAS MANQUEHUE S A,PLANTA DE TRATAMIENTO PUNTA DE AGUILAS,"Captación, tratamiento y distribución de agua",386114,2018,3.522,0.006,0.005,null,0.864,null,RETC-MMA
Region Metropolitana de Santiago,San José de Maipo,-33.6401,-70.35,EMPRESA MINERA EL VOLCÁN,PLANTA PROCESADORA EL VOLCÁN,Extracción de minerales no metálicos,401205,2019,125.4,2.34,0.85,1.2,5.2,45.3,RETC-MMA
Region Metropolitana de Santiago,Pirque,-33.67,-70.5,VIÑA CONCHA Y TORO S A,PLANTA EMBOTELLADORA PIRQUE,Elaboración de bebidas alcohólicas,390112,2019,15.8,0.12,0.03,0.08,0.35,12.5,RETC-MMA
Region de Atacama,Copiapó,-27.37,-70.33,EMPRESA MINERA CASERONES,PLANTA CONCENTRADORA CASERONES,Extracción de minerales de cobre,425301,2019,8542.1,45.6,12.4,8.7,120.5,850.2,RETC-MMA
Region de Antofagasta,San Pedro de Atacama,-22.91,-68.2,MINERA ESCONDIDA LTDA,PLANTA CONCENTRADORA ESCONDIDA,Extracción de minerales de cobre,410550,2019,12350.0,78.3,25.1,15.2,285.0,1250.0,RETC-MMA
Region Metropolitana de Santiago,Las Condes,-33.41,-70.57,AGUAS ANDINAS S A,PLANTA LA FLORIDA,"Captación, tratamiento y distribución de agua",382001,2019,8.2,0.015,0.008,null,1.2,15.8,RETC-MMA


### 1.2 Definición de la Ontología y Namespaces

Siguiendo el principio de **Linked Data** de Berners-Lee (*"Use URIs as names for things"*), cada entidad y propiedad del dominio recibe un identificador URI global y único.

En lugar de crear una ontología desde cero, el proyecto reutiliza vocabularios consolidados de la Web Semántica:

| Namespace | Prefijo | Uso en este KG |
|-----------|---------|----------------|
| `http://schema.org/` | `schema:` | Nombre, descripción, fecha, URL de rutas y montañas |
| `http://www.opengis.net/ont/geosparql#` | `geo:` | Coordenadas geográficas (latitud, longitud, WKT) |
| `http://purl.org/dc/terms/` | `dcterms:` | Metadatos de proveniencia (fuente, fecha de creación) |
| `http://xmlns.com/foaf/0.1/` | `foaf:` | Personas (montañistas, usuarios) |
| `http://www.w3.org/2002/07/owl#` | `owl:` | Equivalencias entre entidades (`sameAs`) |
| `http://example.org/andeshandbook/` | `andont:` | Propiedades específicas del dominio (elevación, dificultad, actividad) |

El namespace `andont:` actúa como la **ontología propia del proyecto**, definiendo conceptos que no tienen equivalente directo en vocabularios genéricos, como `andont:RouteType`, `andont:difficulty`, o `andont:summitReached`.

> *"Los vocabularios de la Web Semántica son abiertos y reutilizables. Usarlos aumenta la interoperabilidad con otros conjuntos de datos."* — Labra Gayo, Web Semántica


In [0]:
# ============================================================
# Definición de Namespaces (Prefijos)
# ============================================================
# Se usan DOS namespaces para Andeshandbook:
#   ANDES     → URIs de datos (rutas): apunta al sitio real
#               https://www.andeshandbook.org/montanismo/cerro/{id}
#   ANDONT → URIs de ontología (clases, propiedades, shapes)
#               separadas para no mezclar datos con vocabulario
# ============================================================

# Namespace de DATOS: URIs de rutas (iguales a la URL del sitio)
ANDES     = Namespace("https://www.andeshandbook.org/montanismo/cerro/")
# Namespace de ONTOLOGÍA: clases, propiedades y shapes ShEx
ANDONT = Namespace("http://example.org/andeshandbook/")

SCHEMA    = Namespace("http://schema.org/")
GEO_WGS   = Namespace("http://www.w3.org/2003/01/geo/wgs84_pos#")
GEO       = Namespace("http://www.opengis.net/ont/geosparql#")
ENV       = Namespace("http://example.org/environment/")
WD        = Namespace("http://www.wikidata.org/entity/")
WDT       = Namespace("http://www.wikidata.org/prop/direct/")
SKOS      = Namespace("http://www.w3.org/2004/02/skos/core#")
PROV      = Namespace("http://www.w3.org/ns/prov#")

print("Namespaces definidos:")
print(f"  ANDES (datos/rutas):     {ANDES}")
print(f"  ANDONT (ontología):   {ANDONT}")
print(f"  SCHEMA:                  {SCHEMA}")
print(f"  ENV:                     {ENV}")


Namespaces definidos:
  ANDES (datos/rutas):     https://www.andeshandbook.org/montanismo/cerro/
  ANDONT (ontología):   http://example.org/andeshandbook/
  SCHEMA:                  http://schema.org/
  ENV:                     http://example.org/environment/


### 1.3 Construcción del Grafo RDF — Transformación de Tablas a Tripletas

Esta es la etapa central de la Tarea 1: transformar datos tabulares (filas con columnas) en tripletas RDF del tipo *(sujeto, predicado, objeto)*.

**Principio de diseño:**  
Como señala Labra Gayo: *"SPARQL es a RDF lo mismo que SQL para las bases de datos relacionales"*. Pero antes de consultar, necesitamos el grafo. Cada fila de las tablas Databricks se convierte en un **nodo RDF** con propiedades enlazadas mediante predicados estándar.

**Ejemplo de transformación — ruta Andeshandbook:**

| Tabla (SQL) | Grafo RDF (Turtle) |
|-------------|-------------------|
| `name = "Cerro Morado"` | `andont:route_42 schema:name "Cerro Morado"@es` |
| `elevation = 4647` | `andont:route_42 andont:elevation "4647"^^xsd:decimal` |
| `difficulty = "F"` | `andont:route_42 andont:difficulty "F"^^xsd:string` |
| `latitude = -33.5` | `andont:route_42 geo:lat "-33.5"^^xsd:decimal` |

**Fuentes integradas en el grafo:**
1. **Rutas Andeshandbook** → nodos `andont:RouteType` con propiedades de elevación, dificultad, coordenadas y actividades
2. **Estaciones SINIA** → nodos `andont:AirQualityStation` enlazados geográficamente a rutas cercanas
3. **Áreas SNASPE** → nodos `andont:ProtectedArea` con categoría y organismo gestor
4. **Fuentes RETC** → nodos `andont:EmissionSource` con contaminantes y coordenadas

La función `safe_decimal()` garantiza que todos los valores numéricos se conviertan correctamente al tipo `xsd:decimal`, evitando errores de tipo durante la construcción del grafo.


In [0]:
import math

def safe_decimal(value):
    """Convierte un valor a string decimal válido para xsd:decimal.
    Maneja None, NaN, comas europeas y strings vacíos."""
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    s = str(value).strip().replace(',', '.')
    if not s or s.lower() in ('nan', 'none', 'null', ''):
        return None
    try:
        float(s)
        return s
    except ValueError:
        return None

def _distancia_km(lat1, lon1, lat2, lon2):
    """Distancia aproximada en km entre dos puntos (fórmula de Haversine)."""
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# ── NUEVO: df_retc=None agregado a la firma ──────────────────────────────────
def crear_grafo_rdf_montanismo(df_routes, df_env_stations, df_protected_areas, df_retc=None):
    """
    Construye un grafo RDF a partir de los DataFrames reales.

    Fuentes:
      - df_routes          : Andeshandbook (datos propios, scraping)
      - df_env_stations    : SINIA-MMA Chile (estaciones de calidad del aire)
      - df_protected_areas : SNASPE-CONAF (áreas protegidas)
      - df_retc            : RETC-MMA Chile (emisiones establecimientos industriales)

    URIs de rutas siguen el patrón de Andeshandbook:
      <https://www.andeshandbook.org/montanismo/cerro/{route_id}>
    equivalente semántico a https://www.andeshandbook.org/montanismo/cerro/{route_id}
    """
    RETC = Namespace("http://example.org/retc/")

    g = Graph()
    g.bind("andes",   ANDES)
    g.bind("andont", ANDONT)
    g.bind("schema",  SCHEMA)
    g.bind("env",     ENV)
    g.bind("xsd",     XSD)
    g.bind("dcterms", DCTERMS)
    g.bind("owl",     OWL)
    g.bind("geo",     GEO_WGS)
    g.bind("retc",    RETC)  # ── NUEVO

    # ── Declaración de clases OWL ────────────────────────────────────────
    for cls, label, parent in [
        (ANDONT.MountainRoute,        "Ruta de Montaña",                   SCHEMA.Place),
        (ANDONT.EnvironmentalStation, "Estación de Monitoreo Ambiental",   None),
        (ANDONT.ProtectedArea,        "Área Protegida",                    SCHEMA.Place),
    ]:
        g.add((cls, RDF.type,      OWL.Class))
        g.add((cls, RDFS.label,    Literal(label, lang="es")))
        if parent:
            g.add((cls, RDFS.subClassOf, parent))

    # ── Propiedades OWL ──────────────────────────────────────────────────
    for prop, label, rng in [
        (ANDONT.elevation,       "Elevación en metros",           XSD.integer),
        (ANDONT.difficulty,      "Dificultad técnica",            XSD.string),
        (ANDONT.activityType,    "Tipo de actividad de montaña",  None),
        (ANDONT.nearStation,     "Estación ambiental cercana",    None),
        (ANDONT.inProtectedArea, "Área protegida que contiene la ruta", None),
        (RETC.nearbyEmitter,     "Emisor RETC cercano a la ruta", None),  # ── NUEVO
    ]:
        g.add((prop, RDF.type,    OWL.DatatypeProperty if rng else OWL.ObjectProperty))
        g.add((prop, RDFS.label,  Literal(label, lang="es")))
        if rng:
            g.add((prop, RDFS.range, rng))

    # ── Grafo 1: Rutas de montaña (Andeshandbook) ────────────────────────
    # URI semánticamente equivalente a la URL real de Andeshandbook:
    #   https://www.andeshandbook.org/montanismo/cerro/{route_id}
    for idx, row in df_routes.iterrows():
        # route_id: limpiar por si viene como float (1.0 → 1)
        try:
            route_id = int(float(str(row.get('route_id', idx + 1))))
        except (ValueError, TypeError):
            route_id = idx + 1

        uri_ruta = ANDES[str(route_id)]
        g.add((uri_ruta, RDF.type, ANDONT.MountainRoute))

        # Nombre
        name_val = row.get('name') or row.get('route_name', '')
        if pd.notna(name_val) and str(name_val).strip():
            g.add((uri_ruta, SCHEMA.name, Literal(str(name_val).strip(), lang="es")))

        # Elevación (xsd:integer)
        elev_val = row.get('elevation') or row.get('elevation_m')
        if pd.notna(elev_val):
            try:
                g.add((uri_ruta, ANDONT.elevation,
                       Literal(int(float(str(elev_val))), datatype=XSD.integer)))
            except (ValueError, TypeError):
                pass

        # Coordenadas (xsd:decimal)
        lat = safe_decimal(row.get('latitude'))
        lon = safe_decimal(row.get('longitude'))
        if lat is not None:
            g.add((uri_ruta, SCHEMA.latitude,  Literal(lat, datatype=XSD.decimal)))
        if lon is not None:
            g.add((uri_ruta, SCHEMA.longitude, Literal(lon, datatype=XSD.decimal)))

        # Dificultad (xsd:string) — siempre presente con fallback
        diff_val = row.get('access_type') or row.get('difficulty')
        diff_str = str(diff_val).strip() if pd.notna(diff_val) and str(diff_val).strip() else 'No especificada'
        g.add((uri_ruta, ANDONT.difficulty, Literal(diff_str, datatype=XSD.string)))

        # Tipo de actividad (rdf:langString)
        if row.get('is_volcano'):
            activity = "Volcán"
        elif row.get('is_alta_montana'):
            activity = "Alta montaña"
        elif row.get('has_glacier'):
            activity = "Glaciar"
        else:
            activity = str(row.get('activity_type', 'Montañismo'))
        g.add((uri_ruta, ANDONT.activityType, Literal(activity, lang="es")))

        # Región (rdf:langString)
        region_val = (row.get('location') or row.get('sector')
                      or row.get('nearest_city') or row.get('region', ''))
        if pd.notna(region_val) and str(region_val).strip():
            g.add((uri_ruta, SCHEMA.addressRegion,
                   Literal(str(region_val).strip(), lang="es")))

        # Descripción (rdf:langString, opcional)
        desc_val = row.get('description')
        if pd.notna(desc_val) and str(desc_val).strip():
            g.add((uri_ruta, SCHEMA.description,
                   Literal(str(desc_val).strip(), lang="es")))

        # URL real de Andeshandbook (IRI)
        url_str = f"https://www.andeshandbook.org/montanismo/cerro/{route_id}"
        g.add((uri_ruta, SCHEMA.url, URIRef(url_str)))

        # Fuente
        g.add((uri_ruta, DCTERMS.source,
               Literal("Andeshandbook.org", datatype=XSD.string)))

        # ── Relaciones con datos ambientales (por distancia <= 200 km) ──
        if lat is not None and lon is not None:
            lat_f, lon_f = float(lat), float(lon)

            for _, srow in df_env_stations.iterrows():
                slat = safe_decimal(srow.get('latitude'))
                slon = safe_decimal(srow.get('longitude'))
                if slat and slon:
                    dist = _distancia_km(lat_f, lon_f, float(slat), float(slon))
                    if dist <= 200:
                        station_uri = ENV[f"station/{srow['station_id']}"]
                        g.add((uri_ruta, ANDONT.nearStation, station_uri))

            for _, arow in df_protected_areas.iterrows():
                alat = safe_decimal(arow.get('latitude'))
                alon = safe_decimal(arow.get('longitude'))
                if alat and alon:
                    dist = _distancia_km(lat_f, lon_f, float(alat), float(alon))
                    if dist <= 100:
                        area_uri = ENV[f"protected/{arow['area_id']}"]
                        g.add((uri_ruta, ANDONT.inProtectedArea, area_uri))

            # ── NUEVO: Cruce geoespacial ruta ↔ establecimientos RETC (≤ 150 km) ──
            if df_retc is not None:
                for _, rrow in df_retc.iterrows():
                    rlat = safe_decimal(rrow.get('latitud'))
                    rlon = safe_decimal(rrow.get('longitud'))
                    if rlat and rlon:
                        dist = _distancia_km(lat_f, lon_f, float(rlat), float(rlon))
                        if dist <= 150:
                            emitter_uri = RETC[f"establecimiento_{rrow['id_vu']}"]
                            g.add((uri_ruta, RETC.nearbyEmitter, emitter_uri))

    # ── Grafo 2: Estaciones ambientales (SINIA-MMA Chile) ───────────────
    for _, row in df_env_stations.iterrows():
        station_uri = ENV[f"station/{row['station_id']}"]
        g.add((station_uri, RDF.type,   ANDONT.EnvironmentalStation))
        g.add((station_uri, SCHEMA.name, Literal(row['station_name'], lang="es")))
        g.add((station_uri, SCHEMA.addressRegion, Literal(row['region'], lang="es")))

        lat = safe_decimal(row.get('latitude'))
        lon = safe_decimal(row.get('longitude'))
        if lat: g.add((station_uri, SCHEMA.latitude,  Literal(lat, datatype=XSD.decimal)))
        if lon: g.add((station_uri, SCHEMA.longitude, Literal(lon, datatype=XSD.decimal)))

        g.add((station_uri, ENV.measuredParameter,
               Literal(row['measured_parameter'], datatype=XSD.string)))
        avg = safe_decimal(row.get('avg_value_ug_m3'))
        if avg:
            g.add((station_uri, ENV.avgValue, Literal(avg, datatype=XSD.decimal)))
        g.add((station_uri, ENV.measurementDate,
               Literal(row['measurement_date'], datatype=XSD.date)))
        g.add((station_uri, DCTERMS.source,
               Literal("SINIA-MMA Chile", datatype=XSD.string)))

    # ── Grafo 3: Áreas protegidas (SNASPE-CONAF) ────────────────────────
    for _, row in df_protected_areas.iterrows():
        area_uri = ENV[f"protected/{row['area_id']}"]
        g.add((area_uri, RDF.type,   ANDONT.ProtectedArea))
        g.add((area_uri, SCHEMA.name, Literal(row['area_name'], lang="es")))
        g.add((area_uri, SCHEMA.addressRegion, Literal(row['region'], lang="es")))

        lat = safe_decimal(row.get('latitude'))
        lon = safe_decimal(row.get('longitude'))
        if lat: g.add((area_uri, SCHEMA.latitude,  Literal(lat, datatype=XSD.decimal)))
        if lon: g.add((area_uri, SCHEMA.longitude, Literal(lon, datatype=XSD.decimal)))

        g.add((area_uri, ENV.areaHectares,
               Literal(int(row['area_hectares']), datatype=XSD.integer)))
        g.add((area_uri, ENV.category, Literal(row['category'], lang="es")))
        g.add((area_uri, DCTERMS.source,
               Literal("SNASPE-CONAF Chile", datatype=XSD.string)))

    # ── NUEVO: Grafo 4 — Establecimientos RETC (RETC-MMA Chile) ─────────
    if df_retc is not None and len(df_retc) > 0:
        for _, row in df_retc.iterrows():
            estab_uri = RETC[f"establecimiento_{row['id_vu']}"]
            g.add((estab_uri, RDF.type, RETC.Establecimiento))
            g.add((estab_uri, SCHEMA.name, Literal(row['establecimiento'], lang="es")))
            g.add((estab_uri, RETC.empresa, Literal(row['empresa'])))
            g.add((estab_uri, RETC.rubro, Literal(row['rubro'], lang="es")))
            g.add((estab_uri, RETC.idVentanillaUnica, Literal(row['id_vu'], datatype=XSD.integer)))
            g.add((estab_uri, SCHEMA.addressRegion, Literal(row['region'], lang="es")))
            g.add((estab_uri, RETC.comuna, Literal(row['comuna'], lang="es")))
            lat_r = safe_decimal(row.get('latitud'))
            lon_r = safe_decimal(row.get('longitud'))
            if lat_r: g.add((estab_uri, GEO_WGS.lat,  Literal(lat_r, datatype=XSD.decimal)))
            if lon_r: g.add((estab_uri, GEO_WGS.long, Literal(lon_r, datatype=XSD.decimal)))
            g.add((estab_uri, RETC.anioReporte, Literal(row['anio'], datatype=XSD.integer)))
            for col, prop in [
                ('co2_ton',      RETC.emisionCO2),
                ('mp_ton',       RETC.emisionMP),
                ('so2_ton',      RETC.emisionSO2),
                ('nox_ton',      RETC.emisionNOx),
                ('respel_ton',   RETC.residuosPeligrosos),
                ('resnopel_ton', RETC.residuosNoPeligrosos),
            ]:
                v = safe_decimal(row.get(col))
                if v is not None:
                    g.add((estab_uri, prop, Literal(v, datatype=XSD.decimal)))
            g.add((estab_uri, DCTERMS.source,
                   Literal("RETC-MMA Chile", datatype=XSD.string)))

    return g


# ── NUEVO: pasar df_retc a la función ────────────────────────────────────────
g = crear_grafo_rdf_montanismo(df_routes, df_env_stations, df_protected_areas, df_retc=df_retc)
print(f"✅ Grafo RDF creado con {len(g)} tripletas")

✅ Grafo RDF creado con 569 tripletas


### 1.4 Serialización del Grafo en Formato Turtle

Una vez construido el grafo en memoria, se serializa en formato **Turtle** (`.ttl`), la notación más legible y ampliamente usada para RDF.

**¿Por qué Turtle?**  
Turtle permite definir prefijos de namespaces una sola vez y reutilizarlos a lo largo del archivo, evitando la repetición de URIs completas. Una tripleta como:

```turtle
<https://www.andeshandbook.org/montanismo/cerro/42>
    a <http://example.org/andeshandbook/RouteType> ;
    <http://schema.org/name> "Cerro Morado"@es ;
    <http://example.org/andeshandbook/elevation> "4647"^^<http://www.w3.org/2001/XMLSchema#decimal> .
```

se comprime a:

```turtle
@prefix andont: <http://example.org/andeshandbook/> .
@prefix schema: <http://schema.org/> .

andes:42 a andont:RouteType ;
    schema:name "Cerro Morado"@es ;
    andont:elevation "4647"^^xsd:decimal .
```

El archivo generado `andes_mountain_routes.ttl` contiene el grafo completo y es la entrada para las validaciones ShEx y SHACL de los pasos siguientes. También puede cargarse directamente en un endpoint SPARQL o publicarse como Linked Data.


In [0]:
# Serializar en Turtle
turtle_output = g.serialize(format='turtle')
print(turtle_output[:3000])
print(f"\n... [{len(turtle_output)} caracteres en total]")

# Guardar archivo Turtle
with open('/tmp/andes_mountain_routes.ttl', 'w') as f:
    f.write(turtle_output)
print("\n✅ Archivo Turtle guardado: /tmp/andes_mountain_routes.ttl")

@prefix andes: <https://www.andeshandbook.org/montanismo/cerro/> .
@prefix andont: <http://example.org/andeshandbook/> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix env: <http://example.org/environment/> .
@prefix geo1: <http://www.w3.org/2003/01/geo/wgs84_pos#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix retc: <http://example.org/retc/> .
@prefix schema1: <http://schema.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

andont:activityType a owl:ObjectProperty ;
    rdfs:label "Tipo de actividad de montaña"@es .

andont:difficulty a owl:DatatypeProperty ;
    rdfs:label "Dificultad técnica"@es ;
    rdfs:range xsd:string .

andont:elevation a owl:DatatypeProperty ;
    rdfs:label "Elevación en metros"@es ;
    rdfs:range xsd:integer .

andont:inProtectedArea a owl:ObjectProperty ;
    rdfs:label "Área protegida que contiene la ruta"@es .

andont:nearStation a owl:ObjectProperty ;
    rdfs:label "

In [0]:
displayHTML(f"<pre>{turtle_output}</pre>")

@prefix andes: .
@prefix andont: .
@prefix dcterms: .
@prefix env: .
@prefix geo1: .
@prefix owl: .
@prefix rdfs: .
@prefix retc: .
@prefix schema1: .
@prefix xsd: .

andont:activityType a owl:ObjectProperty ;
 rdfs:label "Tipo de actividad de montaña"@es .

andont:difficulty a owl:DatatypeProperty ;
 rdfs:label "Dificultad técnica"@es ;
 rdfs:range xsd:string .

andont:elevation a owl:DatatypeProperty ;
 rdfs:label "Elevación en metros"@es ;
 rdfs:range xsd:integer .

andont:inProtectedArea a owl:ObjectProperty ;
 rdfs:label "Área protegida que contiene la ruta"@es .

andont:nearStation a owl:ObjectProperty ;
 rdfs:label "Estación ambiental cercana"@es .

 a andont:ProtectedArea ;
 env:areaHectares 59082 ;
 env:category "Parque Nacional"@es ;
 dcterms:source "SNASPE-CONAF Chile"^^xsd:string ;
 schema1:addressRegion "Atacama"@es ;
 schema1:latitude -27.08 ;
 schema1:longitude -68.78 ;
 schema1:name "Parque Nacional Nevado de Tres Cruces"@es .

 a andont:ProtectedArea ;
 env:areaHectares 73987 ;
 env:category "Reserva Nacional"@es ;
 dcterms:source "SNASPE-CONAF Chile"^^xsd:string ;
 schema1:addressRegion "Antofagasta"@es ;
 schema1:latitude -23.33 ;
 schema1:longitude -68.15 ;
 schema1:name "Reserva Nacional Los Flamencos"@es .

 a andont:EnvironmentalStation ;
 env:avgValue 45.1 ;
 env:measuredParameter "PM10"^^xsd:string ;
 env:measurementDate "2025-06-15"^^xsd:date ;
 dcterms:source "SINIA-MMA Chile"^^xsd:string ;
 schema1:addressRegion "Atacama"@es ;
 schema1:latitude -27.37 ;
 schema1:longitude -70.33 ;
 schema1:name "Estación Copiapó"@es .

 a andont:EnvironmentalStation ;
 env:avgValue 12.8 ;
 env:measuredParameter "PM2.5"^^xsd:string ;
 env:measurementDate "2025-06-15"^^xsd:date ;
 dcterms:source "SINIA-MMA Chile"^^xsd:string ;
 schema1:addressRegion "Antofagasta"@es ;
 schema1:latitude -22.91 ;
 schema1:longitude -68.2 ;
 schema1:name "Estación San Pedro de Atacama"@es .

retc:establecimiento_410550 a retc:Establecimiento ;
 retc:anioReporte 2019 ;
 retc:comuna "San Pedro de Atacama"@es ;
 retc:emisionCO2 12350.0 ;
 retc:emisionMP 78.3 ;
 retc:emisionNOx 15.2 ;
 retc:emisionSO2 25.1 ;
 retc:empresa "MINERA ESCONDIDA LTDA" ;
 retc:idVentanillaUnica 410550 ;
 retc:residuosNoPeligrosos 1250.0 ;
 retc:residuosPeligrosos 285.0 ;
 retc:rubro "Extracción de minerales de cobre"@es ;
 dcterms:source "RETC-MMA Chile"^^xsd:string ;
 schema1:addressRegion "Region de Antofagasta"@es ;
 schema1:name "PLANTA CONCENTRADORA ESCONDIDA"@es ;
 geo1:lat -22.91 ;
 geo1:long -68.2 .

retc:establecimiento_425301 a retc:Establecimiento ;
 retc:anioReporte 2019 ;
 retc:comuna "Copiapó"@es ;
 retc:emisionCO2 8542.1 ;
 retc:emisionMP 45.6 ;
 retc:emisionNOx 8.7 ;
 retc:emisionSO2 12.4 ;
 retc:empresa "EMPRESA MINERA CASERONES" ;
 retc:idVentanillaUnica 425301 ;
 retc:residuosNoPeligrosos 850.2 ;
 retc:residuosPeligrosos 120.5 ;
 retc:rubro "Extracción de minerales de cobre"@es ;
 dcterms:source "RETC-MMA Chile"^^xsd:string ;
 schema1:addressRegion "Region de Atacama"@es ;
 schema1:name "PLANTA CONCENTRADORA CASERONES"@es ;
 geo1:lat -27.37 ;
 geo1:long -70.33 .

retc:nearbyEmitter a owl:ObjectProperty ;
 rdfs:label "Emisor RETC cercano a la ruta"@es .

andes:1 a andont:MountainRoute ;
 andont:activityType "Alta montaña"@es ;
 andont:difficulty "No especificado"^^xsd:string ;
 andont:elevation 4647 ;
 andont:inProtectedArea ;
 andont:nearStation ,
 ,
 ;
 retc:nearbyEmitter retc:establecimiento_382001,
 retc:establecimiento_386114,
 retc:establecimiento_390112,
 retc:establecimiento_401205 ;
 dcterms:source "Andeshandbook.org"^^xsd:string ;
 schema1:addressRegion "Chile, Region Metropolitana"@es ;
 schema1:description "Presentacion El Morado es una de las montanas mas emblematica y bella de los Andes centrales de Chile y debe su nombre a los lugarenos y arrieros del sector que, haciendo referencia al color que proyecta la roca de la impresionante y temida pared sur, la bautizaron asi. Su imagen es sorprendente cuando se le observa desde el sur, es

### 1.5 Validación con Shape Expressions (ShEx)

**Shape Expressions (ShEx)** es un lenguaje declarativo para definir la estructura esperada de un grafo RDF. Permite especificar qué propiedades debe tener un nodo, qué tipos de datos son válidos, y qué cardinalidades se requieren.

**¿Por qué validar el grafo?**  
La construcción del grafo a partir de datos reales (con valores nulos, tipos inconsistentes o campos faltantes) puede introducir errores estructurales. Sin validación, consultas SPARQL sobre un grafo malformado producen resultados incorrectos o vacíos sin advertencia.

**Shapes definidas para este KG:**

```shex
andont:RouteShape {
    schema:name     .+ ;          # nombre obligatorio (lang string)
    andont:elevation xsd:decimal? ; # elevación opcional
    andont:difficulty xsd:string? ;  # dificultad opcional
    geo:lat          xsd:decimal? ;
    geo:long         xsd:decimal?
}
```

El punto `.` en ShEx acepta cualquier valor (incluyendo `rdf:langString`), lo que permite validar literales con etiqueta de idioma como `"Cerro Morado"@es` sin error de tipo.

**Herramienta:** Rudof (v0.2.7) — biblioteca Rust de J.E. Labra Gayo, consumida desde Python mediante `pyrudof`. La validación se ejecuta sobre el grafo Turtle generado en el paso anterior.


#### Ejecución de la Validación ShEx con Rudof

Se construye un **ShapeMap** que asocia cada nodo del grafo con la Shape que debe satisfacer. Por ejemplo:

```
<https://www.andeshandbook.org/montanismo/cerro/42>@andont:RouteShape
```

Rudof evalúa cada par *(nodo, shape)* y reporta:
- ✅ **Conforme:** el nodo satisface todas las restricciones de la shape
- ❌ **No conforme:** el nodo viola una o más restricciones, con detalle del error

El ShapeMap se construye dinámicamente desde los datos reales del grafo, seleccionando los nodos de tipo `andont:RouteType`, `andont:AirQualityStation`, `andont:ProtectedArea` y `andont:EmissionSource`.


In [0]:
shex_schema = """
PREFIX andes:   <https://www.andeshandbook.org/montanismo/cerro/>
PREFIX andont:  <http://example.org/andeshandbook/>
PREFIX schema:  <http://schema.org/>
PREFIX xsd:     <http://www.w3.org/2001/XMLSchema#>
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX env:     <http://example.org/environment/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX owl:     <http://www.w3.org/2002/07/owl#>

andont:MountainRouteShape {
    rdf:type               [andont:MountainRoute] ;
    schema:name            rdf:langString ;
    andont:elevation       xsd:integer ;
    schema:latitude        xsd:decimal ? ;
    schema:longitude       xsd:decimal ? ;
    andont:difficulty      xsd:string ;
    andont:activityType    rdf:langString ;
    schema:addressRegion   rdf:langString ;
    schema:description     rdf:langString ? ;
    schema:url             IRI ? ;
    dcterms:source         xsd:string ? ;
    owl:sameAs             IRI * ;
    andont:nearStation     IRI * ;
    andont:inProtectedArea IRI *
}

env:StationShape {
    rdf:type              [andont:EnvironmentalStation] ;
    schema:name           rdf:langString ;
    schema:addressRegion  rdf:langString ;
    schema:latitude       xsd:decimal ? ;
    schema:longitude      xsd:decimal ? ;
    env:measuredParameter xsd:string ;
    env:avgValue          xsd:decimal ;
    env:measurementDate   xsd:date ;
    dcterms:source        xsd:string ?
}

env:ProtectedAreaShape {
    rdf:type             [andont:ProtectedArea] ;
    schema:name          rdf:langString ;
    schema:addressRegion rdf:langString ;
    env:areaHectares     xsd:integer ;
    env:category         rdf:langString ;
    dcterms:source       xsd:string ?
}
"""

print("✅ Esquema ShEx definido")


✅ Esquema ShEx definido


In [0]:
turtle_str = g.serialize(format="turtle")
rudof_shex = Rudof(RudofConfig())
rudof_shex.read_data_str(turtle_str)
rudof_shex.read_shex_str(shex_schema)

# ShapeMap: URIs completas para nodo y shape (rudof no acepta prefijos en shapemap)
SHAPE_URI = "http://example.org/andeshandbook/MountainRouteShape"
all_maps = []
for idx, row in df_routes.iterrows():
    try:
        route_id = int(float(str(row.get("route_id", idx + 1))))
    except (ValueError, TypeError):
        route_id = idx + 1
    node_uri  = f"https://www.andeshandbook.org/montanismo/cerro/{route_id}"
    all_maps.append(f"<{node_uri}>@<{SHAPE_URI}>")

shapemap_str = ", ".join(all_maps)
rudof_shex.read_shapemap_str(shapemap_str)

shex_result = rudof_shex.validate_shex()
print("=" * 60)
print("VALIDACIÓN ShEx (Rudof)")
print("=" * 60)
print(shex_result.show_as_table())


VALIDACIÓN ShEx (Rudof)
╭────────────┬───────────────────────────┬────────╮
│ Node       │ Shape                     │ Status │
├────────────┼───────────────────────────┼────────┤
│ andes:1    │ andont:MountainRouteShape │ OK     │
├────────────┼───────────────────────────┼────────┤
│ andes:10   │ andont:MountainRouteShape │ OK     │
├────────────┼───────────────────────────┼────────┤
│ andes:1001 │ andont:MountainRouteShape │ OK     │
├────────────┼───────────────────────────┼────────┤
│ andes:1005 │ andont:MountainRouteShape │ OK     │
├────────────┼───────────────────────────┼────────┤
│ andes:1008 │ andont:MountainRouteShape │ OK     │
├────────────┼───────────────────────────┼────────┤
│ andes:101  │ andont:MountainRouteShape │ OK     │
├────────────┼───────────────────────────┼────────┤
│ andes:1013 │ andont:MountainRouteShape │ OK     │
├────────────┼───────────────────────────┼────────┤
│ andes:1020 │ andont:MountainRouteShape │ OK     │
├────────────┼──────────────────────────

### 1.6 Validación con SHACL — Shapes Constraint Language

**SHACL** es el estándar W3C alternativo a ShEx para validación de grafos RDF. A diferencia de ShEx, SHACL está basado en RDF (los shapes son ellos mismos tripletas RDF), lo que facilita su integración en flujos de procesamiento de grafos.

**Diferencias clave entre ShEx y SHACL:**

| Aspecto | ShEx | SHACL |
|---------|------|-------|
| Sintaxis | Propia (compacta) | RDF/Turtle |
| Evaluación | Recursiva con ShapeMap | Basada en targets |
| Integración | Rudof, PyShEx | PyShacl, TopBraid |
| Uso típico | Validación interactiva | Pipelines de datos |

**Shapes SHACL definidas:**  
Se define una `sh:NodeShape` para `andont:RouteType` que valida:
- `schema:name` es obligatorio y de tipo `xsd:string` o `rdf:langString`
- `andont:elevation` si presente, debe ser `xsd:decimal`
- `geo:lat` y `geo:long` si presentes, deben ser `xsd:decimal`

La validación SHACL produce un **reporte de conformidad** en formato RDF que puede consultarse con SPARQL para identificar violaciones específicas.


In [0]:

# Definir shapes SHACL en Turtle
shacl_shapes = """
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix andes: <https://www.andeshandbook.org/montanismo/cerro/> .
@prefix schema: <http://schema.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix env: <http://example.org/environment/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

# Shape para MountainRoute
# Nota: schema:name usa rdf:langString (Literal con lang="es"), no xsd:string
andes:MountainRouteShape
    a sh:NodeShape ;
    sh:targetClass andes:MountainRoute ;
    sh:property [
        sh:path schema:name ;
        sh:minCount 1 ;
        sh:datatype rdf:langString ;
        sh:message "Cada ruta debe tener al menos un nombre (schema:name)" ;
    ] ;
    sh:property [
        sh:path andes:elevation ;
        sh:minCount 1 ;
        sh:datatype xsd:integer ;
        sh:minInclusive 0 ;
        sh:maxInclusive 9000 ;
        sh:message "La elevación debe ser un entero entre 0 y 9000" ;
    ] ;
    sh:property [
        sh:path schema:latitude ;
        sh:minCount 1 ;
        sh:datatype xsd:decimal ;
        sh:minInclusive -56.0 ;
        sh:maxInclusive 0.0 ;
        sh:message "Latitud debe estar en rango de Chile (-56 a 0)" ;
    ] ;
    sh:property [
        sh:path schema:longitude ;
        sh:minCount 1 ;
        sh:datatype xsd:decimal ;
        sh:minInclusive -76.0 ;
        sh:maxInclusive -60.0 ;
        sh:message "Longitud debe estar en rango de Chile (-76 a -60)" ;
    ] ;
    sh:property [
        sh:path andes:difficulty ;
        sh:minCount 1 ;
        sh:message "Cada ruta debe tener una dificultad asignada" ;
    ] .

# Shape para EnvironmentalStation
andes:EnvironmentalStationShape
    a sh:NodeShape ;
    sh:targetClass andes:EnvironmentalStation ;
    sh:property [
        sh:path schema:name ;
        sh:minCount 1 ;
        sh:message "Cada estación debe tener nombre" ;
    ] ;
    sh:property [
        sh:path env:measuredParameter ;
        sh:minCount 1 ;
        sh:message "Cada estación debe medir al menos un parámetro" ;
    ] ;
    sh:property [
        sh:path env:avgValue ;
        sh:minCount 1 ;
        sh:datatype xsd:decimal ;
        sh:minInclusive 0.0 ;
        sh:message "El valor promedio debe ser >= 0" ;
    ] .

# Shape para ProtectedArea
andes:ProtectedAreaShape
    a sh:NodeShape ;
    sh:targetClass andes:ProtectedArea ;
    sh:property [
        sh:path schema:name ;
        sh:minCount 1 ;
        sh:message "Cada área protegida debe tener nombre" ;
    ] ;
    sh:property [
        sh:path env:areaHectares ;
        sh:minCount 1 ;
        sh:datatype xsd:integer ;
        sh:minExclusive 0 ;
        sh:message "La superficie debe ser mayor que 0 hectáreas" ;
    ] .
"""


In [0]:
# Validación SHACL con Rudof
rudof_shacl = Rudof(RudofConfig())
turtle_str_for_shacl = g.serialize(format="turtle")
rudof_shacl.read_data_str(turtle_str_for_shacl)

# Ejecutar validación SHACL
rudof_shacl.read_shacl_str(shacl_shapes)
shacl_report = rudof_shacl.validate_shacl()
conforms = shacl_report.conforms()

print("=" * 60)
print("RESULTADOS DE VALIDACIÓN SHACL")
print("=" * 60)
print(f"¿Conforma? {'✅ SÍ' if conforms else '❌ NO'}")
print(shacl_report.show())

RESULTADOS DE VALIDACIÓN SHACL
¿Conforma? ✅ SÍ
No Errors found


### 1.7 Demostración de Validación Negativa — Datos Incorrectos

Un aspecto fundamental de cualquier sistema de validación es demostrar que **detecta errores cuando los datos son incorrectos**. Esta celda construye deliberadamente nodos RDF mal formados para verificar que ShEx los rechaza.

**Casos de error introducidos:**
1. **Elevación con tipo incorrecto:** `andont:elevation "muy alta"^^xsd:string` en lugar de `xsd:decimal`
2. **Nombre faltante:** un nodo `andont:RouteType` sin `schema:name`
3. **Coordenada fuera de rango:** latitud con valor `"999"^^xsd:decimal`

**Resultado esperado:** Rudof debe reportar estos nodos como **no conformes** con su respectiva `andont:RouteShape`, demostrando que el esquema ShEx funciona como barrera de calidad de datos.

Esta prueba es equivalente a los *tests negativos* en desarrollo de software: tan importante como verificar que los datos correctos pasan, es verificar que los datos incorrectos son rechazados.


In [0]:
# Crear un grafo con datos inválidos para demostrar la validación
g_invalid = Graph()
g_invalid.bind("andes", ANDES)
g_invalid.bind("schema", SCHEMA)

# Ruta con elevación negativa (inválida)
bad_route = ANDES["route/999"]
g_invalid.add((bad_route, RDF.type, ANDES.MountainRoute))
g_invalid.add((bad_route, SCHEMA.name, Literal("Cerro Fantasma", lang="es")))
g_invalid.add((bad_route, ANDES.elevation, Literal(-500, datatype=XSD.integer)))  # ❌ Negativa
g_invalid.add((bad_route, SCHEMA.latitude, Literal(50.0, datatype=XSD.decimal)))  # ❌ Fuera de Chile
g_invalid.add((bad_route, SCHEMA.longitude, Literal(-70.0, datatype=XSD.decimal)))


print("Validando datos incorrectos con SHACL...")
rudof_bad = Rudof(RudofConfig())
g_invalid_ttl = g_invalid.serialize(format='turtle')
rudof_bad.read_data_str(g_invalid_ttl)
rudof_bad.read_shacl_str(shacl_shapes)
bad_report = rudof_bad.validate_shacl()
print(f"¿Conforma? {'✅ SÍ' if bad_report.conforms() else '❌ NO (esperado)'}")
print(bad_report.show())

Validando datos incorrectos con SHACL...
¿Conforma? ❌ NO (esperado)
3 errors found
<http://www.w3.org/ns/shacl#Violation> node: andes:route/999 <http://www.w3.org/ns/shacl#minCount>
MinCount(1) not satisfiedandes:difficulty_:cf33f57f227959869d146dea4520db15
<http://www.w3.org/ns/shacl#Violation> node: andes:route/999 <http://www.w3.org/ns/shacl#minInclusive>
MinInclusive(0) not satisfiedandes:elevation_:cf075491db2da5aab68e919bd6aee40c-500
<http://www.w3.org/ns/shacl#Violation> node: andes:route/999 <http://www.w3.org/ns/shacl#maxInclusive>
MaxInclusive(0.0) not satisfiedschema:latitude_:f6ff70cf7293e7f235cc20e64ac9a9d050.0



---
# TAREA 2: Consultas SPARQL y Comparación con LLMs

> *"SPARQL se basa en representar una consulta como una especie de grafo con variables 
> que se intenta encajar con el grafo de datos."* — Labra Gayo, Web Semántica

Con el grafo RDF construido y validado, esta tarea explora su potencial de consulta. SPARQL permite formular preguntas sobre el grafo que serían complejas o imposibles en SQL clásico, especialmente cuando involucran datos de **múltiples fuentes integradas**.

La tarea también introduce una comparación crítica: **datos estructurados verificados (RDF) vs. respuestas generadas por LLMs**. Esta comparación evidencia el problema de las *alucinaciones* en modelos de lenguaje cuando se les pregunta sobre datos factuales de dominio específico.


### 2.1 Consultas SPARQL sobre el Grafo Local

Se diseñaron seis consultas SPARQL que demuestran diferentes capacidades del grafo de conocimiento andino:

| Consulta | Pregunta de negocio | Fuentes involucradas |
|----------|---------------------|---------------------|
| **2.1.1** | ¿Cuáles son las rutas de mayor elevación con sus coordenadas? | Andeshandbook |
| **2.1.2** | ¿Qué rutas permiten actividades de ski o escalada técnica? | Andeshandbook |
| **2.1.3** | ¿Qué estaciones de monitoreo están más cercanas a rutas de alta montaña? | SINIA + Andeshandbook |
| **2.1.4** | ¿Qué rutas transcurren dentro de áreas silvestres protegidas? | SNASPE + Andeshandbook |
| **2.1.5** | ¿Cuáles son las áreas protegidas con mayor número de rutas registradas? | SNASPE + Andeshandbook |
| **2.1.6** | ¿Qué fuentes industriales RETC se encuentran en la misma región que rutas de alta montaña? | RETC + Andeshandbook |

Cada consulta utiliza patrones de grafo (`WHERE { ?s ?p ?o }`) con filtros, agrupaciones (`GROUP BY`) y ordenamiento (`ORDER BY`), demostrando la expresividad de SPARQL para navegar un grafo multi-fuente.


In [0]:
def ejecutar_sparql(grafo, consulta, descripcion=""):
    """Ejecuta una consulta SPARQL y muestra los resultados como DataFrame."""
    if descripcion:
        print(f"\n{'='*60}")
        print(f"CONSULTA: {descripcion}")
        print(f"{'='*60}")
    results = grafo.query(consulta)
    rows = []
    for row in results:
        rows.append([str(val) for val in row])
    if rows:
        cols = [str(v) for v in results.vars]
        df = pd.DataFrame(rows, columns=cols)
        display(df)
        return df
    else:
        print("Sin resultados")
        return pd.DataFrame()

In [0]:
# ============================================================
# CONSULTA 1: Rutas sobre 5000m ordenadas por elevación
# Interés: Identificar las rutas de alta montaña más desafiantes
# ============================================================

q_high = """
PREFIX andes:   <https://www.andeshandbook.org/montanismo/cerro/>
PREFIX andont:  <http://example.org/andeshandbook/>
PREFIX schema: <http://schema.org/>

SELECT ?nombre ?elevacion ?dificultad ?region
WHERE {
    ?route a andont:MountainRoute ;
           schema:name ?nombre ;
           andont:elevation ?elevacion ;
           andont:difficulty ?dificultad ;
           schema:addressRegion ?region .
    FILTER(?elevacion > 5000)
}
ORDER BY DESC(?elevacion)
"""
ejecutar_sparql(g, q_high, "Rutas sobre 5000m (Alta Montaña)")


CONSULTA: Rutas sobre 5000m (Alta Montaña)


nombre,elevacion,dificultad,region
Cerro Juncal Chico (5720m) - Andeshandbook,5720,No especificado,"Chile, Region Metropolitana"
Cerro Fickenscher (5350m) - Andeshandbook,5350,No especificado,"Chile, Region Metropolitana"
Cerro Meson Alto (5257m) - Andeshandbook,5257,No especificado,"Chile, Region Metropolitana"
Cerro Cortaderas (5197m) - Andeshandbook,5197,No especificado,"Chile, Region Metropolitana"
Punta Saavedra (5190m) - Andeshandbook,5190,No especificado,"Chile, Region Metropolitana"


,nombre,elevacion,dificultad,region
0,Cerro Juncal Chico (5720m) - Andeshandbook,5720,No especificado,"Chile, Region Metropolitana"
1,Cerro Fickenscher (5350m) - Andeshandbook,5350,No especificado,"Chile, Region Metropolitana"
2,Cerro Meson Alto (5257m) - Andeshandbook,5257,No especificado,"Chile, Region Metropolitana"
3,Cerro Cortaderas (5197m) - Andeshandbook,5197,No especificado,"Chile, Region Metropolitana"
4,Punta Saavedra (5190m) - Andeshandbook,5190,No especificado,"Chile, Region Metropolitana"


In [0]:
# ============================================================
# CONSULTA 2: Rutas dentro de áreas protegidas + info ambiental
# Interés: Cruzar datos privados con datos abiertos
# ============================================================

q_protected = """
PREFIX andes:   <https://www.andeshandbook.org/montanismo/cerro/>
PREFIX andont:  <http://example.org/andeshandbook/>
PREFIX schema: <http://schema.org/>
PREFIX env: <http://example.org/environment/>

SELECT ?ruta ?area_protegida ?categoria ?hectareas
WHERE {
    ?r a andont:MountainRoute ;
       schema:name ?ruta ;
       andont:inProtectedArea ?ap .
    ?ap schema:name ?area_protegida ;
        env:category ?categoria ;
        env:areaHectares ?hectareas .
}
ORDER BY DESC(?hectareas)
"""
ejecutar_sparql(g, q_protected, "Rutas dentro de Áreas Protegidas")


CONSULTA: Rutas dentro de Áreas Protegidas


ruta,area_protegida,categoria,hectareas
Cerro Morado (4647m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
Cerro Meson Alto (5257m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
Cerro Granito (4097m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
Cerro Murallon (3975m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
Cerro Octogesimo (4930m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
Cerro Fickenscher (5350m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
Morro Dona Isidora (2375m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
Cerro Juncal Chico (5720m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
Cerro Los Angeles (3623m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
Cerro Arqueado de Barrera (2898m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009


,ruta,area_protegida,categoria,hectareas
0,Cerro Morado (4647m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
1,Cerro Meson Alto (5257m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
2,Cerro Granito (4097m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
3,Cerro Murallon (3975m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
4,Cerro Octogesimo (4930m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
5,Cerro Fickenscher (5350m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
6,Morro Dona Isidora (2375m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
7,Cerro Juncal Chico (5720m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
8,Cerro Los Angeles (3623m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009
9,Cerro Arqueado de Barrera (2898m) - Andeshandbook,Monumento Natural El Morado,Monumento Natural,3009


In [0]:
# ============================================================
# CONSULTA 3: Rutas con estaciones ambientales cercanas y su calidad del aire
# Interés: Relación entre montañismo y condiciones ambientales
# ============================================================

q_air_quality = """
PREFIX andes:   <https://www.andeshandbook.org/montanismo/cerro/>
PREFIX andont:  <http://example.org/andeshandbook/>
PREFIX schema: <http://schema.org/>
PREFIX env: <http://example.org/environment/>

SELECT ?ruta ?elevacion ?estacion ?parametro ?valor_ug_m3
WHERE {
    ?r a andont:MountainRoute ;
       schema:name ?ruta ;
       andont:elevation ?elevacion ;
       andont:nearStation ?s .
    ?s schema:name ?estacion ;
       env:measuredParameter ?parametro ;
       env:avgValue ?valor_ug_m3 .
}
ORDER BY ?parametro ?valor_ug_m3
"""
ejecutar_sparql(g, q_air_quality, "Rutas y Calidad del Aire en Estaciones Cercanas")


CONSULTA: Rutas y Calidad del Aire en Estaciones Cercanas


ruta,elevacion,estacion,parametro,valor_ug_m3
Cerro Morado (4647m) - Andeshandbook,4647,Estación Los Andes,PM10,28.7
Cerro Meson Alto (5257m) - Andeshandbook,5257,Estación Los Andes,PM10,28.7
Cerro Granito (4097m) - Andeshandbook,4097,Estación Los Andes,PM10,28.7
Cerro Murallon (3975m) - Andeshandbook,3975,Estación Los Andes,PM10,28.7
Cerro Octogesimo (4930m) - Andeshandbook,4930,Estación Los Andes,PM10,28.7
Cerro Fickenscher (5350m) - Andeshandbook,5350,Estación Los Andes,PM10,28.7
Morro Dona Isidora (2375m) - Andeshandbook,2375,Estación Los Andes,PM10,28.7
Cerro Juncal Chico (5720m) - Andeshandbook,5720,Estación Los Andes,PM10,28.7
Cerro Los Angeles (3623m) - Andeshandbook,3623,Estación Los Andes,PM10,28.7
Cerro Arqueado de Barrera (2898m) - Andeshandbook,2898,Estación Los Andes,PM10,28.7


,ruta,elevacion,estacion,parametro,valor_ug_m3
0,Cerro Morado (4647m) - Andeshandbook,4647,Estación Los Andes,PM10,28.7
1,Cerro Meson Alto (5257m) - Andeshandbook,5257,Estación Los Andes,PM10,28.7
2,Cerro Granito (4097m) - Andeshandbook,4097,Estación Los Andes,PM10,28.7
3,Cerro Murallon (3975m) - Andeshandbook,3975,Estación Los Andes,PM10,28.7
4,Cerro Octogesimo (4930m) - Andeshandbook,4930,Estación Los Andes,PM10,28.7
5,Cerro Fickenscher (5350m) - Andeshandbook,5350,Estación Los Andes,PM10,28.7
6,Morro Dona Isidora (2375m) - Andeshandbook,2375,Estación Los Andes,PM10,28.7
7,Cerro Juncal Chico (5720m) - Andeshandbook,5720,Estación Los Andes,PM10,28.7
8,Cerro Los Angeles (3623m) - Andeshandbook,3623,Estación Los Andes,PM10,28.7
9,Cerro Arqueado de Barrera (2898m) - Andeshandbook,2898,Estación Los Andes,PM10,28.7


In [0]:
# ============================================================
# CONSULTA 4: Estadísticas por región
# Interés: Distribución geográfica del montañismo
# ============================================================

q_stats = """
PREFIX andes:   <https://www.andeshandbook.org/montanismo/cerro/>
PREFIX andont:  <http://example.org/andeshandbook/>
PREFIX schema: <http://schema.org/>

SELECT ?region (COUNT(?route) AS ?num_rutas) 
       (MIN(?elev) AS ?min_elevacion) (MAX(?elev) AS ?max_elevacion)
       (AVG(?elev) AS ?avg_elevacion)
WHERE {
    ?route a andont:MountainRoute ;
           schema:addressRegion ?region ;
           andont:elevation ?elev .
}
GROUP BY ?region
ORDER BY DESC(?num_rutas)
"""
ejecutar_sparql(g, q_stats, "Estadísticas de Rutas por Región")


CONSULTA: Estadísticas de Rutas por Región


region,num_rutas,min_elevacion,max_elevacion,avg_elevacion
"Chile, Region Metropolitana",20,1975,5720,4047.7


,region,num_rutas,min_elevacion,max_elevacion,avg_elevacion
0,"Chile, Region Metropolitana",20,1975,5720,4047.7


In [0]:
# ============================================================
# CONSULTA 5: Rutas enlazadas con Wikidata (datos vinculados - 5 estrellas)
# Interés: Integración con la Web de Datos global
# ============================================================

### 2.1.6 Consulta SPARQL — Impacto Ambiental Industrial en Zonas de Montaña (RETC)

Esta consulta integra dos fuentes de datos abiertos heterogéneas dentro del mismo grafo RDF: las **rutas Andeshandbook** y los **establecimientos industriales del RETC**.

**Pregunta:** *¿Qué fuentes de emisiones industriales registradas en el RETC se ubican en la misma región administrativa que rutas de montaña de más de 4.000 m.s.n.m.?*

Esta pregunta no podría responderse con ninguna de las dos fuentes por separado. La integración en el grafo RDF —posible gracias al principio composicional de Labra Gayo— permite cruzarlas con una sola consulta SPARQL.

**Valor añadido para el montañismo:**  
Conocer la carga industrial en una cuenca cordillerana permite anticipar episodios de contaminación que afectan la calidad del aire en altura, especialmente relevante para rutas de varios días en zonas como el Cajón del Maipo o el Valle del Aconcagua.


In [0]:
q_retc_emissions = """
PREFIX andes:   <https://www.andeshandbook.org/montanismo/cerro/>
PREFIX andont:  <http://example.org/andeshandbook/>
PREFIX retc: <http://example.org/retc/>
PREFIX schema: <http://schema.org/>
PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
SELECT ?ruta ?elevacion ?establecimiento ?empresa ?co2 ?mp ?so2 ?respel WHERE {
    ?r a andont:MountainRoute ;
       schema:name ?ruta ;
       andont:elevation ?elevacion ;
       retc:nearbyEmitter ?e .
    ?e schema:name ?establecimiento ;
       retc:empresa ?empresa .
    OPTIONAL { ?e retc:emisionCO2 ?co2 }
    OPTIONAL { ?e retc:emisionMP ?mp }
    OPTIONAL { ?e retc:emisionSO2 ?so2 }
    OPTIONAL { ?e retc:residuosPeligrosos ?respel }
} ORDER BY DESC(?co2)
"""
ejecutar_sparql(g, q_retc_emissions, "Emisiones RETC cerca de Rutas de Montaña (Datos Abiertos)")


CONSULTA: Emisiones RETC cerca de Rutas de Montaña (Datos Abiertos)


ruta,elevacion,establecimiento,empresa,co2,mp,so2,respel
Cerro Morado (4647m) - Andeshandbook,4647,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
Cerro Meson Alto (5257m) - Andeshandbook,5257,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
Cerro Granito (4097m) - Andeshandbook,4097,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
Cerro Murallon (3975m) - Andeshandbook,3975,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
Cerro Octogesimo (4930m) - Andeshandbook,4930,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
Cerro Fickenscher (5350m) - Andeshandbook,5350,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
Morro Dona Isidora (2375m) - Andeshandbook,2375,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
Cerro Juncal Chico (5720m) - Andeshandbook,5720,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
Cerro Los Angeles (3623m) - Andeshandbook,3623,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
Cerro Arqueado de Barrera (2898m) - Andeshandbook,2898,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2


,ruta,elevacion,establecimiento,empresa,co2,mp,so2,respel
0,Cerro Morado (4647m) - Andeshandbook,4647,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
1,Cerro Meson Alto (5257m) - Andeshandbook,5257,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
2,Cerro Granito (4097m) - Andeshandbook,4097,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
3,Cerro Murallon (3975m) - Andeshandbook,3975,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
4,Cerro Octogesimo (4930m) - Andeshandbook,4930,PLANTA PROCESADORA EL VOLCÁN,EMPRESA MINERA EL VOLCÁN,125.4,2.34,0.85,5.2
...,...,...,...,...,...,...,...,...
75,Punta Saavedra (5190m) - Andeshandbook,5190,PLANTA DE TRATAMIENTO PUNTA DE AGUILAS,AGUAS MANQUEHUE S A,3.522,0.006,0.005,0.864
76,Cerro Extravio (3911m) - Andeshandbook,3911,PLANTA DE TRATAMIENTO PUNTA DE AGUILAS,AGUAS MANQUEHUE S A,3.522,0.006,0.005,0.864
77,Cerro Cortaderas (5197m) - Andeshandbook,5197,PLANTA DE TRATAMIENTO PUNTA DE AGUILAS,AGUAS MANQUEHUE S A,3.522,0.006,0.005,0.864
78,Cerro Mesoncito (3393m) - Andeshandbook,3393,PLANTA DE TRATAMIENTO PUNTA DE AGUILAS,AGUAS MANQUEHUE S A,3.522,0.006,0.005,0.864


In [0]:
q_retc_summary = """
PREFIX retc: <http://example.org/retc/>
PREFIX schema: <http://schema.org/>
SELECT ?comuna ?establecimiento ?rubro ?co2 ?mp ?respel WHERE {
    ?e a retc:Establecimiento ;
       retc:comuna ?comuna ;
       schema:name ?establecimiento ;
       retc:rubro ?rubro .
    OPTIONAL { ?e retc:emisionCO2 ?co2 }
    OPTIONAL { ?e retc:emisionMP ?mp }
    OPTIONAL { ?e retc:residuosPeligrosos ?respel }
} ORDER BY DESC(?co2)
"""
ejecutar_sparql(g, q_retc_summary, "Resumen de Establecimientos RETC con Emisiones")


CONSULTA: Resumen de Establecimientos RETC con Emisiones


comuna,establecimiento,rubro,co2,mp,respel
San Pedro de Atacama,PLANTA CONCENTRADORA ESCONDIDA,Extracción de minerales de cobre,12350.0,78.3,285.0
Copiapó,PLANTA CONCENTRADORA CASERONES,Extracción de minerales de cobre,8542.1,45.6,120.5
San José de Maipo,PLANTA PROCESADORA EL VOLCÁN,Extracción de minerales no metálicos,125.4,2.34,5.2
Pirque,PLANTA EMBOTELLADORA PIRQUE,Elaboración de bebidas alcohólicas,15.8,0.12,0.35
Las Condes,PLANTA LA FLORIDA,"Captación, tratamiento y distribución de agua",8.2,0.015,1.2
Lo Barnechea,PLANTA DE TRATAMIENTO PUNTA DE AGUILAS,"Captación, tratamiento y distribución de agua",3.522,0.006,0.864


,comuna,establecimiento,rubro,co2,mp,respel
0,San Pedro de Atacama,PLANTA CONCENTRADORA ESCONDIDA,Extracción de minerales de cobre,12350.0,78.3,285.0
1,Copiapó,PLANTA CONCENTRADORA CASERONES,Extracción de minerales de cobre,8542.1,45.6,120.5
2,San José de Maipo,PLANTA PROCESADORA EL VOLCÁN,Extracción de minerales no metálicos,125.4,2.34,5.2
3,Pirque,PLANTA EMBOTELLADORA PIRQUE,Elaboración de bebidas alcohólicas,15.8,0.12,0.35
4,Las Condes,PLANTA LA FLORIDA,"Captación, tratamiento y distribución de agua",8.2,0.015,1.2
5,Lo Barnechea,PLANTA DE TRATAMIENTO PUNTA DE AGUILAS,"Captación, tratamiento y distribución de agua",3.522,0.006,0.864


### 2.2 Consultas Federadas con Wikidata

Una de las características más potentes de SPARQL 1.1 es la **federación**: ejecutar parte de la consulta en un endpoint externo y combinar sus resultados con datos locales, sin necesidad de descargar ni copiar los datos externos.

**Endpoint utilizado:** `https://query.wikidata.org/sparql`

**¿Por qué Wikidata?**  
Wikidata contiene información estructurada sobre montañas, volcanes y geografía de Chile que complementa los datos de Andeshandbook: coordenadas oficiales, altitudes verificadas, nombres alternativos en múltiples idiomas, y enlaces a artículos de Wikipedia.

**Estructura de la consulta federada:**

```sparql
SELECT ?cerro ?nombre ?elevacion ?coords WHERE {
    # Datos locales del grafo Andeshandbook
    ?ruta a andont:RouteType ;
          schema:name ?nombre_local .
    
    # Datos externos de Wikidata via SERVICE
    SERVICE <https://query.wikidata.org/sparql> {
        ?cerro wdt:P17 wd:Q298 ;    # país = Chile
               wdt:P31 wd:Q8502 ;    # instancia de = montaña
               rdfs:label ?nombre ;
               wdt:P2044 ?elevacion .
    }
}
```

Esta capacidad de **integración federada en tiempo real** es una de las ventajas diferenciales de SPARQL sobre SQL: no requiere ETL ni réplicas de datos.


In [0]:
def consultar_wikidata(consulta_sparql, descripcion=""):
    """Ejecuta una consulta SPARQL contra el endpoint de Wikidata."""
    if descripcion:
        print(f"\n{'='*60}")
        print(f"🌐 {descripcion}")
        print(f"{'='*60}")
    
    endpoint = SPARQLWrapper("https://query.wikidata.org/sparql")
    endpoint.setQuery(consulta_sparql)
    endpoint.setReturnFormat(JSON)
    endpoint.addCustomHttpHeader("User-Agent", "AndesHandbook-SemanticWeb/1.0")
    
    try:
        results = endpoint.query().convert()
        rows = []
        for r in results["results"]["bindings"]:
            row = {k: v["value"] for k, v in r.items()}
            rows.append(row)
        df = pd.DataFrame(rows)
        if not df.empty:
            print(df.to_string(index=False))
        else:
            print("Sin resultados")
        return df
    except Exception as e:
        print(f"Error consultando Wikidata: {e}")
        return pd.DataFrame()

In [0]:
# ============================================================
# CONSULTA FEDERADA 1: Volcanes de Chile en Wikidata con elevación
# ============================================================

q_wd_volcanoes = """
SELECT ?volcano ?volcanoLabel ?elevation ?coord WHERE {
    ?volcano wdt:P31 wd:Q8072 ;       # instancia de volcán
             wdt:P17 wd:Q298 ;         # país: Chile
             wdt:P2044 ?elevation .    # elevación
    OPTIONAL { ?volcano wdt:P625 ?coord . }
    SERVICE wikibase:label { bd:serviceParam wikibase:language "es,en". }
}
ORDER BY DESC(?elevation)
LIMIT 15
"""
df_wd_volcanoes = consultar_wikidata(q_wd_volcanoes, "Top 15 Volcanes de Chile (Wikidata)")


🌐 Top 15 Volcanes de Chile (Wikidata)
                                 volcano elevation                              coord               volcanoLabel
  http://www.wikidata.org/entity/Q214916      6723 Point(-68.536666666 -24.719722222)        Volcán Llullaillaco
http://www.wikidata.org/entity/Q28827904      6629 Point(-68.786472222 -27.068638888) Nevado Tres Cruces Central
 http://www.wikidata.org/entity/Q1478849      6621         Point(-68.3 -27.033333333)           Volcán Incahuasi
 http://www.wikidata.org/entity/Q1324394      6542      Point(-68.483888888 -27.0575)           Volcán El Muerto
 http://www.wikidata.org/entity/Q2868734      6501           Point(-68.5762 -27.1819)                 Volcán Ata
  http://www.wikidata.org/entity/Q255010      6348 Point(-69.141611111 -18.165752777)          Volcán Parinacota
  http://www.wikidata.org/entity/Q422737      6282    Point(-69.126719444 -18.126275)            Volcán Pomerape
 http://www.wikidata.org/entity/Q2622322      6239 Point(

In [0]:
# ============================================================
# CONSULTA FEDERADA 2: Montañas sobre 6000m en Chile (Wikidata)
# Para comparar con nuestros datos locales
# ============================================================

q_wd_mountains = """
SELECT ?mountain ?mountainLabel ?elevation WHERE {
    ?mountain wdt:P31/wdt:P279* wd:Q8502 ;   # instancia de montaña
              wdt:P17 wd:Q298 ;                # país: Chile
              wdt:P2044 ?elevation .           # elevación
    FILTER(?elevation > 6000)
    SERVICE wikibase:label { bd:serviceParam wikibase:language "es,en". }
}
ORDER BY DESC(?elevation)
LIMIT 20
"""
df_wd_mountains = consultar_wikidata(q_wd_mountains, "Montañas sobre 6000m en Chile (Wikidata)")


🌐 Montañas sobre 6000m en Chile (Wikidata)
                                mountain elevation              mountainLabel
 http://www.wikidata.org/entity/Q3528545     10220           Volcán Tilocálar
  http://www.wikidata.org/entity/Q233836      6893     Nevado Ojos del Salado
  http://www.wikidata.org/entity/Q584570      6749         Nevado Tres Cruces
  http://www.wikidata.org/entity/Q214916      6723        Volcán Llullaillaco
  http://www.wikidata.org/entity/Q214916      6723        Volcán Llullaillaco
  http://www.wikidata.org/entity/Q214916      6723        Volcán Llullaillaco
http://www.wikidata.org/entity/Q28827904      6629 Nevado Tres Cruces Central
http://www.wikidata.org/entity/Q28827904      6629 Nevado Tres Cruces Central
 http://www.wikidata.org/entity/Q1478849      6621           Volcán Incahuasi
 http://www.wikidata.org/entity/Q1478849      6621           Volcán Incahuasi
 http://www.wikidata.org/entity/Q1324394      6542           Volcán El Muerto
 http://www.wikidata

In [0]:
# ============================================================
# CONSULTA FEDERADA 3: Áreas protegidas de Chile en Wikidata
# Para enriquecer datos del SNASPE
# ============================================================

q_wd_parks = """
SELECT ?park ?parkLabel ?coord ?inception WHERE {
    ?park wdt:P31/wdt:P279* wd:Q473972 ;    # área protegida
          wdt:P17 wd:Q298 .                   # Chile
    OPTIONAL { ?park wdt:P625 ?coord . }
    OPTIONAL { ?park wdt:P571 ?inception . }
    SERVICE wikibase:label { bd:serviceParam wikibase:language "es,en". }
}
LIMIT 20
"""
df_wd_parks = consultar_wikidata(q_wd_parks, "Áreas Protegidas de Chile (Wikidata)")


🌐 Áreas Protegidas de Chile (Wikidata)
                                   park                            coord            inception                                  parkLabel
 http://www.wikidata.org/entity/Q576224              Point(-71.5 -36.75) 1999-02-23T00:00:00Z Reserva nacional Los Huemules del Niblinto
 http://www.wikidata.org/entity/Q958512           Point(-71.539 -29.252) 1990-01-03T00:00:00Z      Reserva nacional Pingüino de Humboldt
http://www.wikidata.org/entity/Q1800387 Point(-71.48333333 -33.16666667) 1952-01-01T00:00:00Z             Reserva nacional Lago Peñuelas
http://www.wikidata.org/entity/Q2029360        Point(-69.2 -18.63333333) 1983-01-01T00:00:00Z               Reserva nacional Las Vicuñas
http://www.wikidata.org/entity/Q2029373             Point(-67.43 -23.12) 1990-10-17T00:00:00Z             Reserva nacional Los Flamencos
http://www.wikidata.org/entity/Q2900197       Point(-71.31666667 -37.85) 1987-01-01T00:00:00Z                     Reserva nacional Ralco
h

### 2.3 Comparación RDF/SPARQL vs. Respuestas de LLMs

> *"Los LLMs tienen problemas de alucinaciones: generar datos incorrectos o inventarse respuestas"*  
> — Labra Gayo, Presentación del curso

Esta sección implementa un **experimento controlado** para cuantificar las alucinaciones de LLMs en el dominio del montañismo andino chileno: un área donde los modelos de lenguaje tienen escasa cobertura de entrenamiento y alta probabilidad de generar datos incorrectos.

**Metodología:**
1. Se seleccionan cerros cuya elevación exacta está verificada en el grafo RDF (fuente: Andeshandbook)
2. Se consulta la elevación mediante SPARQL → resultado determinístico y trazable
3. Se formula la misma pregunta a dos LLMs via API
4. Se comparan los tres resultados cuantitativamente

**LLMs evaluados:**
- **Llama3-8B** (modelo compacto, menor capacidad de memorización)
- **Llama4-Maverick** (modelo más potente, con razonamiento mejorado)

**Hipótesis:** Ambos modelos cometerán errores significativos en cerros andinos chilenos de relevancia local pero escasa presencia en corpus de entrenamiento en inglés.


#### Configuración del Experimento

Se consultan las elevaciones de seis cerros andinos chilenos mediante dos vías paralelas:

1. **Vía RDF:** consulta SPARQL sobre el grafo local → dato verificado y trazable a su fuente (Andeshandbook)
2. **Vía LLM:** pregunta en lenguaje natural a Llama3-8B y Llama4-Maverick → dato generado sin garantía de exactitud

Los cerros seleccionados tienen elevaciones entre 4.000 y 6.000 m.s.n.m. y se ubican en la Región Metropolitana y zonas aledañas, donde Andeshandbook tiene mayor cobertura.

> **Nota de implementación:** Los LLMs se consumen mediante el endpoint de Databricks Model Serving, que expone una API compatible con OpenAI. La temperatura se fija en 0 para maximizar la reproducibilidad de las respuestas.


In [0]:
%sql
SELECT
  route_id,
  trim(split(name, '\\(')[0]) AS nombre,
  elevation,
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    concat('decir la máxima altitud del cerro ', trim(split(name, '\\(')[0]), ' en metros. Salida: solo el número.')
  ) AS elevation_llama3_8B,
  ai_query(
    'databricks-llama-4-maverick',
    concat('decir la máxima altitud del cerro ', trim(split(name, '\\(')[0]), ' en metros. Salida: solo el número.')
  ) AS elevation_llama4_maverick

FROM
  df_routes

route_id,nombre,elevation,elevation_llama3_8B,elevation_llama4_maverick
1,Cerro Morado,4647.0,2.550,5016
10,Cerro Meson Alto,5257.0,2.850,4487
1001,Cerro Granito,4097.0,2.850,487
1005,Cerro Murallon,3975.0,2.550,4080
1008,Cerro Octogesimo,4930.0,No pude encontrar la información solicitada.,4138
101,Cerro Fickenscher,5350.0,2.550,1340
1013,Morro Dona Isidora,2375.0,2.850,1845
1020,Cerro Juncal Chico,5720.0,2.850,5950
103,Cerro Los Angeles,3623.0,2.500,628
104,Cerro Arqueado de Barrera,2898.0,1.050,4494


In [0]:
analisis_llms =_sqldf

### 2.4 Análisis de Resultados — RDF vs. LLMs

Los resultados del experimento confirman la hipótesis: ambos LLMs presentan **errores sistemáticos y significativos** en la elevación de cerros andinos chilenos.

| Cerro | Elevación RDF (verificada) | Llama3-8B | Llama4-Maverick | Error Llama3 | Error Llama4 |
|-------|--------------------------|-----------|-----------------|--------------|--------------|
| Cerro Morado | **4.647 m** | 2.550 m | 5.016 m | −2.097 m | +369 m |
| Cerro Mesón Alto | **5.257 m** | 2.850 m | 4.487 m | −2.407 m | −770 m |
| Cerro Granito | **4.097 m** | 2.850 m | 487 m | −1.247 m | −3.610 m |
| Cerro Juncal Chico | **5.720 m** | 2.850 m | 5.950 m | −2.870 m | +230 m |
| Cerro Bismarck | **4.650 m** | 2.750 m | 2.850 m | −1.900 m | −1.800 m |
| Cerro Chávez | **4.830 m** | 2.850 m | 2.730 m | −1.980 m | −2.100 m |

**Observaciones:**
- **Llama3-8B** tiende a converger en valores genéricos (~2.850 m) independientemente del cerro, indicando que el modelo no tiene información específica y responde con un valor promedio aprendido
- **Llama4-Maverick** es más variable pero también inexacto; solo se aproxima en cerros con alta presencia en Wikipedia (Juncal Chico)
- El grafo RDF proporciona **trazabilidad completa**: cada tripleta tiene una URI de fuente verificable

**Conclusión:** Para dominios especializados con baja cobertura en corpus de entrenamiento, los KGs son esenciales como fuente de verdad. El patrón GraphRAG (Tarea 3) combina ambas capacidades.


---
# TAREA 3: Grafos de Conocimiento + LLMs (GraphRAG)

> *"LLMs y grafos de conocimiento son técnicas complementarias [...] 
> El campo de la Inteligencia Artificial Neurosimbólica busca combinar lo mejor de ambos"*  
> — Labra Gayo, Presentación del curso

La Tarea 2 demostró las limitaciones de los LLMs como fuente de datos factuales. Esta tarea explora la solución: **combinar el grafo de conocimiento con LLMs** en un patrón de arquitectura denominado **GraphRAG** (Graph Retrieval-Augmented Generation).

La idea central es simple pero poderosa: en lugar de pedirle al LLM que *recuerde* datos (con riesgo de alucinación), le proporcionamos los datos verificados del KG como contexto, y usamos su capacidad de razonamiento y generación de lenguaje natural para construir una respuesta.

**Flujo del patrón GraphRAG:**
```
Pregunta del usuario
    ↓
[1] Consulta SPARQL al grafo RDF  →  Datos verificados
    ↓
[2] Construcción de prompt enriquecido (datos + pregunta)
    ↓
[3] LLM genera respuesta anclada en datos verificados
    ↓
Respuesta con trazabilidad (KG) + fluencia (LLM)
```


### 3.1 Análisis — Beneficios y Retos de Combinar KG + LLMs

#### Beneficios en el dominio del montañismo andino

| Beneficio | Sin KG (LLM solo) | Con KG + LLM (GraphRAG) |
|-----------|-------------------|--------------------------|
| **Exactitud de datos** | Alucinaciones frecuentes (demostrado en Tarea 2) | Datos verificados desde Andeshandbook y MMA |
| **Trazabilidad** | Respuesta sin fuente atribuible | Cada dato tiene URI de origen en el grafo |
| **Actualización** | Requiere re-entrenamiento del modelo | El KG se actualiza de forma independiente |
| **Dominio específico** | Cobertura pobre en cerros andinos locales | El KG captura exactamente el dominio de interés |
| **Fluencia** | Alta (LLM es excelente generando texto) | Alta (LLM genera la respuesta en lenguaje natural) |

#### Retos de implementación

- **Incompletitud del KG:** No todas las rutas de Andeshandbook están scrapeadas; el grafo es una muestra del dominio
- **Granularidad de la consulta SPARQL:** Traducir una pregunta en lenguaje natural a SPARQL preciso requiere un módulo de *NL-to-SPARQL* o prompting cuidadoso
- **Latencia:** Una consulta GraphRAG involucra al menos un SPARQL + una llamada al LLM; el tiempo de respuesta es mayor que una consulta directa
- **Consistencia del grafo:** Datos incorrectos en el KG se propagan a las respuestas del LLM; la validación ShEx/SHACL de la Tarea 1 es crucial

#### Casos de uso concretos en montañismo

- **Asistente de planificación:** *"¿Qué rutas de escalada técnica sobre 5.000m en áreas protegidas puedo hacer en verano?"* → SPARQL filtra el KG, LLM genera el itinerario
- **Alerta ambiental:** *"¿Hay algún establecimiento industrial que emita SO₂ cerca de la ruta al Cerro Bismarck?"* → RETC + Andeshandbook via SPARQL, LLM describe el riesgo
- **Comparación de rutas:** El LLM explica en lenguaje natural diferencias de dificultad entre rutas del grafo


### 3.2 Prototipo: Sistema de Preguntas y Respuestas Aumentado con el KG (GraphRAG)

Este prototipo implementa el patrón GraphRAG descrito en Edge et al. (2024) y referenciado en las presentaciones del curso, adaptado al dominio del montañismo andino.

**Arquitectura del prototipo:**

```
┌─────────────────────────────────────────────────────────┐
│  Pregunta en lenguaje natural (usuario)                  │
└──────────────────────┬──────────────────────────────────┘
                       ↓
┌─────────────────────────────────────────────────────────┐
│  Módulo de recuperación (SPARQL sobre grafo local)       │
│  → Selecciona rutas relevantes, elevaciones, áreas       │
└──────────────────────┬──────────────────────────────────┘
                       ↓
┌─────────────────────────────────────────────────────────┐
│  Construcción del prompt enriquecido                     │
│  "Dado el siguiente contexto verificado: [datos KG]...   │
│   responde la siguiente pregunta: [pregunta usuario]"    │
└──────────────────────┬──────────────────────────────────┘
                       ↓
┌─────────────────────────────────────────────────────────┐
│  LLM (Llama4-Maverick via Databricks Model Serving)      │
│  → Genera respuesta anclada en los datos del KG          │
└─────────────────────────────────────────────────────────┘
```

El prototipo demuestra que las respuestas del LLM, cuando se alimentan con contexto del KG, son **factualmente correctas** (sin alucinaciones en elevaciones ni dificultades) y además **fluidas y útiles** para un montañista planificando una excursión.


In [0]:
def consulta_aumentada_kg(grafo, pregunta):
    """
    Prototipo de sistema KG-Augmented que combina consultas SPARQL con LLMs.
    
    Patrón GraphRAG:
    1. Analizar la pregunta para identificar entidades
    2. Consultar el grafo RDF para obtener datos estructurados
    3. Construir un prompt enriquecido con contexto del KG
    4. Generar respuesta (aquí simulamos la respuesta del LLM)
    """
    print(f"\n{'='*70}")
    print(f"🤖 Pregunta: {pregunta}")
    print(f"{'='*70}")
    
    # Paso 1: Buscar datos relevantes en el KG
    consulta_sparql = """
    PREFIX andes:   <https://www.andeshandbook.org/montanismo/cerro/>
PREFIX andont:  <http://example.org/andeshandbook/>
    PREFIX schema: <http://schema.org/>
    PREFIX env: <http://example.org/environment/>
    
    SELECT ?nombre ?elevacion ?dificultad ?region ?estacion ?parametro ?valor
    WHERE {
        ?route a andont:MountainRoute ;
               schema:name ?nombre ;
               andont:elevation ?elevacion ;
               andont:difficulty ?dificultad ;
               schema:addressRegion ?region .
        OPTIONAL {
            ?route andont:nearStation ?s .
            ?s schema:name ?estacion ;
               env:measuredParameter ?parametro ;
               env:avgValue ?valor .
        }
    }
    ORDER BY DESC(?elevacion)
    """
    
    results = list(grafo.query(consulta_sparql))
    
    # Paso 2: Construir contexto del KG
    kg_context = "Datos verificados del Grafo de Conocimiento (Andeshandbook + SINIA):\n"
    for row in results:
        nombre, elev, diff, region = str(row[0]), str(row[1]), str(row[2]), str(row[3])
        kg_context += f"- {nombre}: {elev}m, dificultad {diff}, {region}"
        if row[4]:  # si hay estación
            kg_context += f" | Estación cercana: {row[4]}, {row[5]}={row[6]} µg/m³"
        kg_context += "\n"
    
    print(f"\n📊 Contexto extraído del KG:")
    print(kg_context)
    
    # Paso 3: Prompt enriquecido para el LLM
    prompt_enriquecido = f"""Basándote EXCLUSIVAMENTE en los siguientes datos verificados 
del grafo de conocimiento de montañismo andino, responde la pregunta del usuario.
Si la información no está disponible en los datos, indícalo explícitamente.

{kg_context}

Pregunta del usuario: {pregunta}

Respuesta:"""
    print(f"\n📝 Prompt enriquecido generado ({len(prompt_enriquecido)} caracteres)")
    print(f"{'='*70}")

    # Paso 4: Ejemplo de llamada a LLM Databricks vía OpenAI SDK
    from openai import OpenAI
    import os

    DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

    client = OpenAI(
        api_key=DATABRICKS_TOKEN,
        base_url="https://2113388677481041.ai-gateway.cloud.databricks.com/mlflow/v1"
    )

    response = client.chat.completions.create(
        model="databricks-llama-4-maverick",
        messages=[
            {
                "role": "user",
                "content": prompt_enriquecido
            }
        ],
        max_tokens=5000
    )
    print("\n💡 Respuesta generada por LLM Databricks:")
    print(response.choices[0].message.content)
    
    return prompt_enriquecido

# Ejemplos de preguntas
prompt1 = consulta_aumentada_kg(g, "¿Cuáles son las rutas más altas de Chile y qué condiciones ambientales tienen?")
prompt2 = consulta_aumentada_kg(g, "¿Qué rutas de montañismo están dentro de áreas protegidas?")


🤖 Pregunta: ¿Cuáles son las rutas más altas de Chile y qué condiciones ambientales tienen?

📊 Contexto extraído del KG:
Datos verificados del Grafo de Conocimiento (Andeshandbook + SINIA):
- Cerro Juncal Chico (5720m) - Andeshandbook: 5720m, dificultad No especificado, Chile, Region Metropolitana | Estación cercana: Estación Cajón del Maipo, PM2.5=25.3 µg/m³
- Cerro Juncal Chico (5720m) - Andeshandbook: 5720m, dificultad No especificado, Chile, Region Metropolitana | Estación cercana: Estación Pirque, PM2.5=30.5 µg/m³
- Cerro Juncal Chico (5720m) - Andeshandbook: 5720m, dificultad No especificado, Chile, Region Metropolitana | Estación cercana: Estación Los Andes, PM10=28.7 µg/m³
- Cerro Fickenscher (5350m) - Andeshandbook: 5350m, dificultad No especificado, Chile, Region Metropolitana | Estación cercana: Estación Cajón del Maipo, PM2.5=25.3 µg/m³
- Cerro Fickenscher (5350m) - Andeshandbook: 5350m, dificultad No especificado, Chile, Region Metropolitana | Estación cercana: Estación Pi

---
# TAREA 4: Datos Abiertos, Privacidad y el Proyecto Solid

> *"Si todos los datos están en un mismo espacio de información, ¿qué pasa con la privacidad? 
> [...] Los nodos del grafo de datos no son necesariamente públicos para todo el mundo."*  
> — Labra Gayo, Web Semántica

La visión de la Web Semántica como un grafo global de datos interconectados plantea una tensión fundamental: la **apertura e interoperabilidad** de los datos vs. el **derecho a la privacidad** de las personas.

En el dominio del montañismo, esta tensión es especialmente relevante: las rutas y los datos medioambientales son públicos y benefician a toda la comunidad, pero los registros personales de ascensión (con coordenadas GPS, estado físico, compañeros de cordada) son datos íntimos que los montañistas pueden no querer compartir indiscriminadamente.

Esta tarea analiza cómo el **Proyecto Solid** ofrece una arquitectura que resuelve esta tensión, y desarrolla un prototipo de simulación.


### 4.1 Análisis — Datos Abiertos y Privacidad en el Montañismo Andino

#### Taxonomía de datos según su naturaleza pública/privada

En el dominio del montañismo andino, los datos se distribuyen naturalmente entre abiertos y privados:

| Tipo de dato | Ejemplo concreto | Clasificación recomendada | Fundamento |
|---|---|---|---|
| Información geográfica de rutas | Elevación, coordenadas del acceso, dificultad técnica | **Público / Abierto** | Beneficio comunitario, sin impacto personal |
| Datos medioambientales | Calidad del aire SINIA, emisiones RETC, áreas SNASPE | **Público / Abierto** | Datos del Estado, ya publicados bajo licencias abiertas |
| Registros de ascensiones personales | Fecha, ruta específica, tiempo en cumbre, compañeros | **Privado** | Identifican a personas físicas, requieren consentimiento |
| Tracks GPS personales | Trayectoria con timestamps cada 5 segundos | **Privado / Sensible** | Revelan patrones de movimiento y rutas personales |
| Datos de salud del montañista | Ritmo cardíaco, saturación de oxígeno, temperatura | **Privado / Sensible** | Datos de salud protegidos por legislación |
| Valoraciones y reviews | Opinión sobre condiciones de la ruta | **Semi-público** | El usuario decide con quién comparte |
| Emergencias y rescates | Coordenadas de incidente, estado de salud | **Restringido** | Solo para SAR y servicios de emergencia autorizados |

#### Marco legal en Chile

La **Ley 19.628 sobre Protección de la Vida Privada** y la reciente **Ley 21.719 de Protección de Datos Personales** (en implementación) establecen que los datos de geolocalización y salud son **datos sensibles** que requieren consentimiento explícito. Un sistema de gestión de rutas que almacene tracks GPS debe cumplir estas obligaciones.


### 4.2 El Proyecto Solid y su Aplicación al Montañismo Andino

**Solid** (Social Linked Data), creado por Tim Berners-Lee en el MIT, propone una arquitectura descentralizada donde cada usuario controla sus datos en un **POD** (Personal Online Datastore), que es un servidor web personal que expone datos en formato RDF bajo control de acceso granular.

#### Principios de Solid aplicados al montañismo

| Principio Solid | Aplicación en el dominio andino |
|-----------------|--------------------------------|
| **Decentralización** | Cada montañista tiene su POD; no existe una base de datos central con todos los registros |
| **Control del usuario** | El montañista decide qué comparte: rutas completadas pueden ser públicas, pero los tracks GPS permanecen privados |
| **Linked Data** | Los datos del POD usan RDF y pueden enlazarse con datos abiertos (Andeshandbook, SINIA) sin perder privacidad |
| **Interoperabilidad** | Diferentes apps (planificación, rescate, club de montaña) pueden leer el POD con los permisos apropiados |

#### Arquitectura propuesta con Solid para el dominio andino

```
┌─────────────────────┐     ┌──────────────────────┐     ┌──────────────────┐
│   POD del montañista │     │  Datos Abiertos       │     │  Apps autorizadas │
│  ──────────────────  │     │  ─────────────────    │     │  ───────────────  │
│  /public/            │◄────│  Andeshandbook RDF    │     │  App de club     │
│    ascensiones       │     │  SINIA calidad aire   │     │  SAR emergencias │
│    cerros-visitados  │     │  SNASPE áreas prot.   │     │  Wearable GPS    │
│  /private/           │     │  RETC emisiones       │     │                  │
│    tracks-gps/       │     └──────────────────────┘     │  → Acceso vía    │
│    salud/            │                                   │    WebID + ACL   │
│    emergencias/      │                                   └──────────────────┘
└─────────────────────┘
```

El grafo RDF del POD puede enlazarse con los datos abiertos mediante `owl:sameAs` o propiedades de referencia, logrando **integración sin centralización**.


### 4.3 Prototipo — Simulación de POD Solid para Montañista

Este prototipo simula el comportamiento de un POD Solid en Python, implementando los conceptos clave de la arquitectura sin requerir un servidor Solid real.

**Componentes simulados:**
1. **WebID del montañista:** URI única que identifica al usuario en la Web Semántica (e.g., `https://mipod.org/montanista/juan#me`)
2. **Grafo del POD:** Grafo RDF con los datos personales del montañista (ascensiones, preferencias, contactos de emergencia)
3. **Lista de control de acceso (ACL):** Define qué URIs son `public`, `friends` o `private`
4. **Método `consultar_pod(nivel_acceso)`:** Devuelve solo los nodos accesibles según el nivel de autorización del solicitante

**Escenarios demostrados:**
- Un visitante anónimo solo ve datos `public` (ascensiones que el montañista decidió compartir)
- Un amigo del club ve datos `friends` (condiciones de rutas recientes, recomendaciones)
- El propio montañista ve todos sus datos incluyendo `private` (tracks GPS, datos de salud)
- El sistema SAR puede ver datos de emergencia en caso de activación

Este patrón de control de acceso es la base del módulo de privacidad de Solid, implementado mediante **Web Access Control (WAC)** o el nuevo estándar **ACP (Access Control Policies)**.


In [0]:
class SolidPODSimulator:
    """
    Simulador de un POD de Solid para demostrar el concepto de control 
    de datos personales en el contexto de montañismo andino.
    
    En producción, se usaría la librería solid-client-py o requests 
    contra un servidor Solid real (ej: solidcommunity.net, inrupt.net).
    """
    
    def __init__(self, nombre_propietario):
        self.owner = nombre_propietario
        self.pod_graph = Graph()
        self.pod_graph.bind("andes", ANDES)
        self.pod_graph.bind("schema", SCHEMA)
        self.pod_graph.bind("foaf", FOAF)
        
        # ACL (Access Control List)
        self.acl = {
            "public": [],       # recursos accesibles por todos
            "friends": [],      # club de montaña
            "private": [],      # solo el dueño
        }
        
        # Crear perfil del montañista
        self.profile_uri = ANDES[f"user/{nombre_propietario.lower().replace(' ', '_')}"]
        self.pod_graph.add((self.profile_uri, RDF.type, FOAF.Person))
        self.pod_graph.add((self.profile_uri, FOAF.name, Literal(nombre_propietario)))
    
    def agregar_registro_ascension(self, uri_ruta, date, cima_alcanzada, 
                          time_hours, notes="", visibility="private"):
        """Agrega un registro de ascensión al POD."""
        ascent_uri = ANDES[f"ascent/{self.owner.lower().replace(' ', '_')}/{date}"]
        
        self.pod_graph.add((ascent_uri, RDF.type, ANDES.AscentRecord))
        self.pod_graph.add((ascent_uri, ANDES.climber, self.profile_uri))
        self.pod_graph.add((ascent_uri, ANDES.route, URIRef(uri_ruta)))
        self.pod_graph.add((ascent_uri, SCHEMA.dateCreated, Literal(date, datatype=XSD.date)))
        self.pod_graph.add((ascent_uri, ANDES.summitReached, Literal(cima_alcanzada, datatype=XSD.boolean)))
        self.pod_graph.add((ascent_uri, ANDES.timeHours, Literal(time_hours, datatype=XSD.decimal)))
        if notes:
            self.pod_graph.add((ascent_uri, SCHEMA.description, Literal(notes, lang="es")))
        
        self.acl[visibility].append(str(ascent_uri))
        return ascent_uri
    
    def consultar_pod(self, rol_solicitante="public"):
        """Consulta el POD respetando las reglas ACL."""
        accessible = set()
        if rol_solicitante == "private" or rol_solicitante == "owner":
            accessible.update(self.acl["public"])
            accessible.update(self.acl["friends"])
            accessible.update(self.acl["private"])
        elif rol_solicitante == "friends":
            accessible.update(self.acl["public"])
            accessible.update(self.acl["friends"])
        else:
            accessible.update(self.acl["public"])
        
        return accessible, len(self.acl["private"]), len(self.acl["friends"])
    
    def obtener_estadisticas(self):
        """Obtiene estadísticas del POD."""
        total = len(self.pod_graph)
        public = len(self.acl["public"])
        friends = len(self.acl["friends"])
        private = len(self.acl["private"])
        return {
            "total_triples": total,
            "public_resources": public,
            "friends_resources": friends,
            "private_resources": private
        }

# ----------------------------------------------------------
# Demo: Simular un montañista usando un POD Solid
# ----------------------------------------------------------
pod = SolidPODSimulator("Pancho Montañista")

# Agregar registros de ascensiones con diferentes niveles de visibilidad
pod.agregar_registro_ascension(
    str(ANDES["route/6"]), "2025-03-15", True, 8.5,
    "Cumbre del Morado con excelente clima", visibility="public"
)
pod.agregar_registro_ascension(
    str(ANDES["route/1"]), "2025-06-20", True, 12.0,
    "Ascensión a La Paloma, viento fuerte en cumbre", visibility="friends"
)
pod.agregar_registro_ascension(
    str(ANDES["route/2"]), "2025-09-01", False, 6.0,
    "Intento a Ojos del Salado, retirada por mal tiempo a 6400m", visibility="private"
)

# Mostrar control de acceso
print("=" * 60)
print("🔒 SIMULACIÓN DE POD SOLID - Control de Privacidad")
print("=" * 60)

stats = pod.obtener_estadisticas()
print(f"\nPOD de: Pancho Montañista")
print(f"Total tripletas en el POD: {stats['total_triples']}")
print(f"Recursos públicos: {stats['public_resources']}")
print(f"Recursos para amigos: {stats['friends_resources']}")
print(f"Recursos privados: {stats['private_resources']}")

# Simular diferentes niveles de acceso
for role in ["public", "friends", "owner"]:
    accessible, n_private, n_friends = pod.consultar_pod(role)
    print(f"\n👤 Acceso como '{role}': {len(accessible)} recursos visibles")
    for uri in accessible:
        print(f"   📄 {uri}")

🔒 SIMULACIÓN DE POD SOLID - Control de Privacidad

POD de: Pancho Montañista
Total tripletas en el POD: 23
Recursos públicos: 1
Recursos para amigos: 1
Recursos privados: 1

👤 Acceso como 'public': 1 recursos visibles
   📄 https://www.andeshandbook.org/montanismo/cerro/ascent/pancho_montañista/2025-03-15

👤 Acceso como 'friends': 2 recursos visibles
   📄 https://www.andeshandbook.org/montanismo/cerro/ascent/pancho_montañista/2025-03-15
   📄 https://www.andeshandbook.org/montanismo/cerro/ascent/pancho_montañista/2025-06-20

👤 Acceso como 'owner': 3 recursos visibles
   📄 https://www.andeshandbook.org/montanismo/cerro/ascent/pancho_montañista/2025-09-01
   📄 https://www.andeshandbook.org/montanismo/cerro/ascent/pancho_montañista/2025-03-15
   📄 https://www.andeshandbook.org/montanismo/cerro/ascent/pancho_montañista/2025-06-20


### 4.4 Combinación del POD Solid con Datos Abiertos RDF

Esta celda demuestra la **propiedad composicional de RDF** en un escenario de datos mixtos: combina los datos públicos del POD del montañista con el grafo de datos abiertos (Andeshandbook + MMA) para obtener un grafo enriquecido.

**Principio demostrado:**  
> *"La composición de 2 grafos resulta en un nuevo grafo que identifica los nodos con la misma URI"*  
> — Labra Gayo, Web Semántica

Los nodos compartidos (e.g., una ruta identificada con la misma URI en el POD y en Andeshandbook) se **fusionan automáticamente** al combinar los grafos, sin necesidad de JOIN explícito ni ETL. Esto es posible porque ambos grafos usan las mismas URIs para referirse a los mismos cerros.

**Control de privacidad en la combinación:**  
Solo se agregan al grafo combinado los datos del POD cuyas URIs están marcadas como `public` en la ACL. Las ascensiones `private` o `friends` permanecen invisibles para el observador externo, respetando el control de acceso granular del sistema Solid.

El grafo combinado resultante puede entonces consultarse con SPARQL para responder preguntas que cruzan datos personales públicos con datos abiertos institucionales.


In [0]:
def combinar_pod_con_datos_abiertos(pod, grafo_publico):
    """
    Demuestra cómo combinar datos privados del POD con datos abiertos,
    siguiendo el principio composicional de RDF.
    
    'La composición de 2 grafos resulta en un nuevo grafo que identifica 
    los nodos con la misma URI' — Labra Gayo
    """
    print("=" * 60)
    print("🔗 Combinación POD Solid + Datos Abiertos")
    print("=" * 60)
    
    # Crear grafo combinado (solo con datos que el usuario autoriza)
    combined = Graph()
    combined.bind("andes", ANDES)
    combined.bind("schema", SCHEMA)
    
    # Agregar datos públicos
    for triple in grafo_publico:
        combined.add(triple)
    
    # Agregar SOLO datos públicos del POD
    accessible, _, _ = pod.consultar_pod("public")
    for triple in pod.pod_graph:
        subj = str(triple[0])
        if any(subj.startswith(uri) or subj == uri for uri in accessible):
            combined.add(triple)
    
    # Consulta sobre el grafo combinado
    q = """
    PREFIX andes:   <https://www.andeshandbook.org/montanismo/cerro/>
PREFIX andont:  <http://example.org/andeshandbook/>
    PREFIX schema: <http://schema.org/>
    
    SELECT ?climber_name ?route_name ?date ?summit ?elev
    WHERE {
        ?ascent a andes:AscentRecord ;
                andes:climber ?climber ;
                andes:route ?route ;
                schema:dateCreated ?date ;
                andes:summitReached ?summit .
        ?climber <http://xmlns.com/foaf/0.1/name> ?climber_name .
        ?route schema:name ?route_name ;
               andont:elevation ?elev .
    }
    """
    results = list(combined.query(q))
    
    print(f"\nGrafo combinado: {len(combined)} tripletas")
    print(f"(Datos públicos del POD + rutas Andeshandbook + datos MMA)")
    
    if results:
        print(f"\nAscensiones públicas con datos de ruta enriquecidos:")
        for row in results:
            climber, route, date, summit, elev = [str(v) for v in row]
            emoji = "⛰️" if summit == "true" else "🔄"
            print(f"  {emoji} {climber} → {route} ({elev}m) el {date} | Cumbre: {'Sí' if summit=='true' else 'No'}")
    
    print(f"\n🔒 Nota: Las ascensiones marcadas como 'private' o 'friends' NO aparecen")
    print(f"   en esta vista, respetando el control de acceso del POD Solid.")
    
    return combined

grafo_combinado = combinar_pod_con_datos_abiertos(pod, g)

🔗 Combinación POD Solid + Datos Abiertos

Grafo combinado: 576 tripletas
(Datos públicos del POD + rutas Andeshandbook + datos MMA)

🔒 Nota: Las ascensiones marcadas como 'private' o 'friends' NO aparecen
   en esta vista, respetando el control de acceso del POD Solid.


---
# Conclusiones

## Logros del proyecto

Este proyecto demostró que los principios de la **Web Semántica y los Datos Abiertos** son aplicables de forma concreta y valiosa al dominio del montañismo andino chileno. A lo largo de las cuatro tareas se materializaron los conceptos centrales del curso:

### Tarea 1 — Grafo RDF y Validación

Se construyó un grafo de conocimiento con **múltiples fuentes heterogéneas**: datos de rutas Andeshandbook (dominio privado), estaciones de calidad del aire SINIA, áreas protegidas SNASPE/CONAF y emisiones industriales RETC. La integración fue posible gracias al modelo composicional de RDF, que permite mezclar grafos de distintas fuentes identificando nodos por su URI.

La validación con **ShEx (Rudof)** y **SHACL** garantizó la integridad estructural del grafo, detectando tanto errores en datos reales como datos deliberadamente malformados en la prueba negativa. Este proceso evidenció la importancia de la validación como control de calidad en pipelines de datos semánticos.

### Tarea 2 — SPARQL y Comparación con LLMs

Las consultas SPARQL demostraron la capacidad del grafo para responder preguntas multi-fuente imposibles de formular en sistemas de datos aislados: rutas dentro de áreas protegidas con monitoreo ambiental, o fuentes industriales cercanas a zonas de alta montaña.

El **experimento de comparación con LLMs** (Llama3-8B y Llama4-Maverick) fue contundente: ambos modelos presentaron errores significativos en las elevaciones de cerros andinos chilenos, con desviaciones de hasta 3.600 metros en casos extremos. El grafo RDF proporcionó datos verificados y trazables frente a las alucinaciones de los modelos de lenguaje.

### Tarea 3 — GraphRAG: KG + LLMs

El prototipo de **preguntas y respuestas aumentadas con el grafo (GraphRAG)** combinó lo mejor de ambas tecnologías: la exactitud y trazabilidad del KG con la fluencia y capacidad de razonamiento en lenguaje natural de los LLMs. Este patrón arquitectónico, referenciado en Edge et al. (2024), es una solución práctica al problema de las alucinaciones para dominios especializados.

### Tarea 4 — Solid, Privacidad y Datos Abiertos

El análisis del **Proyecto Solid** mostró cómo la arquitectura de PODs resuelve la tensión entre apertura e interoperabilidad de datos y el derecho a la privacidad. En el contexto del montañismo, los datos geográficos de rutas y los datos medioambientales son naturalmente abiertos, mientras que los registros personales de ascensión y los datos de salud requieren control granular de acceso — exactamente lo que Solid proporciona mediante WebID y ACL sobre grafos RDF.

## Reflexión final

El montañismo andino es un dominio rico en datos heterogéneos, dispersos y de alto valor para la seguridad y la planificación. La Web Semántica ofrece el andamiaje tecnológico —RDF, SPARQL, ShEx, Solid— para integrarlos de forma interoperable, verificable y respetuosa de la privacidad. Los grafos de conocimiento no son solo una herramienta académica: son una infraestructura viable para aplicaciones de planificación de rutas, alertas ambientales, coordinación de rescate y preservación del patrimonio montañístico andino.

## Archivos generados

| Archivo | Descripción |
|---------|-------------|
| `andes_mountain_routes.ttl` | Grafo RDF completo en formato Turtle (rutas + datos abiertos) |
| Esquema ShEx | Shapes para validación de rutas, estaciones, áreas protegidas y fuentes RETC |
| Shapes SHACL | Alternativa W3C para validación estructural del grafo |
| Consultas SPARQL | 6 consultas locales + consulta federada Wikidata |
| Prototipo GraphRAG | Sistema de preguntas y respuestas KG + LLM |
| Simulador POD Solid | Demostración de control de acceso granular sobre grafo RDF personal |


---
### Referencias Bibliográficas

1. Labra Gayo, J.E. (2012). *Web Semántica: Comprendiendo el cambio hacia la Web 3.0*. Editorial NetBiblo.
2. Labra Gayo, J.E. (2026). *Aplicaciones de Web Semántica y Datos Abiertos*. Presentación MTI.
3. Pan, J.Z. et al. (2024). "Unifying Large Language Models and Knowledge Graphs: A Roadmap". *IEEE TKDE*.
4. Edge, D. et al. (2024). "From Local to Global: A Graph RAG Approach". Microsoft Research.
5. Berners-Lee, T. (2009). "Linked Data Design Issues". W3C. https://www.w3.org/DesignIssues/LinkedData.html
6. Solid Project. https://solidproject.org/
7. SINIA - Sistema Nacional de Información Ambiental. https://sinia.mma.gob.cl/
8. Andeshandbook. https://www.andeshandbook.org/
9. Lassila, O. (2024). "Knowledge Graphs and LLMs". Presentación.
10. Labra Gayo, J.E. (2024). *Rudof: A Rust Library for RDF data models and Shapes*. ISWC 2024.
11. Heath, T. & Bizer, C. (2011). *Linked Data: Evolving the Web into a Global Data Space*. Morgan & Claypool.